In [1]:
# Parameters
run_date = "2026-01-01"  # papermill replacement
import os
output_dir = os.environ.get("ORION_SIGNALS_DIR", "../signals")
config_path = os.environ.get("DATUM_API_CONFIG_PATH", "../ops/datum_api_config.json")
dry_run = False

# ensure output exists
os.makedirs(output_dir, exist_ok=True)


In [2]:
# Import basic modules
import pandas as pd
from datum_api_client import DatumApi
import datetime
from datetime import timedelta
from typing import Optional, List, Dict, Any


# Import warnings
import warnings
warnings.filterwarnings("ignore")
# pip install xlrd
# pip install openpyxl

ModuleNotFoundError: No module named 'datum_api_client'

In [ ]:
from __future__ import annotations

# ── PairFlux stage 1: shuffle final.parquet into one file per benchmark ───────────────────
#
# Why a shuffle at all: final.parquet is sorted by ticker, but PairFlux needs every ticker of
# one benchmark ALIGNED ON THE SAME TIMESTAMPS. Streaming ticker-by-ticker (the OpenDoor /
# DayTwo pattern) cannot do that, and loading the whole file to pivot it is not an option at
# this universe size. So: one sequential pass writes a small per-benchmark parquet holding
# only [ticker, sdate, smin, stack], already cropped to the three class windows. Stage 2 then
# reads one benchmark at a time and pivots it, which is what makes the memory bounded.
#
# Re-run stage 2 with different thresholds as often as you like — the shuffle is the slow
# part and only has to be redone when the source data or the class windows change.

CLASS_WINDOWS_DEFAULT = {
    "PRE":   ((21, 0), (9, 30)),   # crosses midnight
    "OPEN":  ((9, 0), (10, 0)),    # deliberately overlaps the tail of PRE
    # INTRA opens at 09:45, so its last quarter hour is shared with OPEN. The overlap is the
    # same kind OPEN already has with PRE: a divergence alive at 09:50 is counted by both
    # classes, each measuring it inside its OWN window, and the two ratings stay independent.
    "INTRA": ((9, 45), (16, 0)),
}


def _to_smin(hm, session_split_min):
    """Session minutes. Rows at/after session_split_min belong to the NEXT session day, so
    they are numbered NEGATIVE (21:00 -> -180) and the whole 21:00 -> 16:00 span becomes one
    monotonically increasing axis. Without this the PRE window would wrap around midnight and
    every overnight episode would be cut in half."""
    t = hm[0] * 60 + hm[1]
    return t - 24 * 60 if t >= session_split_min else t


def pairflux_stage1_shuffle(
    input_path: str,
    stage_dir: str,
    *,
    class_windows: dict = None,
    session_split_min: int = 1020,        # 17:00
    start_date: Optional[str] = None,     # "YYYY-MM-DD", session date, inclusive
    bench_whitelist: Optional[List[str]] = None,
    STOCK_NUM_FIELD: str = "Stack%",
    log_every_n_chunks: int = 20,
):
    import gc, time, shutil
    import numpy as np
    import pandas as pd
    import pyarrow as pa
    import pyarrow.parquet as pq
    from pathlib import Path

    if class_windows is None:
        class_windows = CLASS_WINDOWS_DEFAULT

    bounds = [(_to_smin(a, session_split_min), _to_smin(b, session_split_min))
              for a, b in class_windows.values()]
    smin_lo = min(lo for lo, _ in bounds)
    smin_hi = max(hi for _, hi in bounds)

    start_i = int(start_date.replace("-", "")) if start_date else -1

    stage = Path(stage_dir)
    if stage.exists():
        shutil.rmtree(stage)
    stage.mkdir(parents=True, exist_ok=True)

    schema = pa.schema([
        ("ticker", pa.string()),
        ("sdate", pa.int32()),
        ("smin", pa.int16()),
        ("stack", pa.float32()),
    ])
    writers = {}
    counts = {}

    def _writer(bench):
        if bench not in writers:
            safe = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in str(bench))
            writers[bench] = pq.ParquetWriter(str(stage / f"{safe}.parquet"), schema,
                                              compression="zstd")
            counts[bench] = 0
        return writers[bench]

    t0 = time.time()
    total_in = total_out = 0
    pf = pq.ParquetFile(input_path)
    wanted = ["ticker", "dt", "bench", STOCK_NUM_FIELD]
    cols = [c for c in wanted if c in pf.schema.names]
    missing = set(wanted) - set(cols)
    if missing:
        raise KeyError(f"final.parquet is missing required columns: {sorted(missing)}")

    print(f"START PairFlux stage1  file={input_path}")
    print(f"  session_split={session_split_min}min  smin window=[{smin_lo}, {smin_hi}]  start_date={start_date}")

    try:
        for ci in range(pf.num_row_groups):
            df = pf.read_row_group(ci, columns=cols).to_pandas()
            total_in += len(df)

            dt = pd.to_datetime(df["dt"], errors="coerce", utc=True)
            ok = dt.notna().to_numpy(copy=False)
            if not ok.any():
                continue
            dt = dt[ok]
            df = df.loc[ok]

            t_arr = (dt.dt.hour.to_numpy(dtype="int32", copy=False) * 60 +
                     dt.dt.minute.to_numpy(dtype="int32", copy=False))
            late = t_arr >= session_split_min
            smin = np.where(late, t_arr - 24 * 60, t_arr).astype("int16")
            # a row after the split belongs to TOMORROW's session
            sess = dt + pd.to_timedelta(np.where(late, 1, 0), unit="D")
            sdate = (sess.dt.year.to_numpy(dtype="int32", copy=False) * 10000 +
                     sess.dt.month.to_numpy(dtype="int32", copy=False) * 100 +
                     sess.dt.day.to_numpy(dtype="int32", copy=False)).astype("int32")

            stack = pd.to_numeric(df[STOCK_NUM_FIELD], errors="coerce").to_numpy(dtype="float32", copy=False)

            keep = (smin >= smin_lo) & (smin <= smin_hi) & np.isfinite(stack)
            if start_i > 0:
                keep &= sdate >= start_i
            if not keep.any():
                continue

            out = pd.DataFrame({
                "ticker": df["ticker"].to_numpy(copy=False)[keep].astype(str),
                "sdate": sdate[keep],
                "smin": smin[keep],
                "stack": stack[keep],
                "bench": df["bench"].to_numpy(copy=False)[keep],
            })
            out = out[pd.notna(out["bench"])]
            out["bench"] = out["bench"].astype(str).str.strip().str.upper()
            out = out[out["bench"] != ""]
            if bench_whitelist:
                wl = {str(b).strip().upper() for b in bench_whitelist}
                out = out[out["bench"].isin(wl)]
            if out.empty:
                continue

            for bench, part in out.groupby("bench", sort=False):
                tbl = pa.Table.from_pandas(part[["ticker", "sdate", "smin", "stack"]],
                                           schema=schema, preserve_index=False)
                _writer(bench).write_table(tbl)
                counts[bench] += len(part)
                total_out += len(part)

            del df, out
            if (ci + 1) % log_every_n_chunks == 0:
                el = time.time() - t0
                print(f"[rg {ci+1:>4}/{pf.num_row_groups}] in={total_in:,} staged={total_out:,} "
                      f"benches={len(writers)} elapsed={el:.1f}s")
                gc.collect()
    finally:
        for w in writers.values():
            w.close()

    print(f"DONE stage1 in={total_in:,} staged={total_out:,} elapsed={time.time()-t0:.1f}s")
    for b, n in sorted(counts.items(), key=lambda kv: -kv[1]):
        print(f"  {b:<10} rows={n:,}")
    return {b: str(stage / f"{b}.parquet") for b in counts}

In [ ]:
# ── PairFlux stage 2: per-benchmark pair scan ─────────────────────────────────────────────


def pairflux_stats_exporter(
    stage_dir: str,
    *,
    output_onefile_jsonl: str = "PAIRFLUX/onefile.jsonl",
    output_summary_csv: str = "PAIRFLUX/summary.csv",
    output_best_pairs_jsonl: str = "PAIRFLUX/best_pairs.jsonl",
    # one line per divergence episode — the only file that can answer "what happened on
    # 2026-07-14 for this pair"; summary/onefile carry all-history aggregates only.
    output_episodes_jsonl: str = "PAIRFLUX/episodes.jsonl",
    write_episodes: bool = True,
    class_windows: dict = None,
    # ONSET windows: a class may only COUNT divergences that were born inside this narrower
    # slice, while still using the full class window to look for the convergence. OPEN is
    # the motivating case: "did the deviations that appeared between 9:00 and 9:25 normalise
    # by 10:00" — a divergence starting at 9:45 is a different question and must not be
    # mixed into the same rate. Classes absent from this dict use their full window.
    onset_windows: dict = None,          # {"OPEN": ((9, 0), (9, 25))}
    # An episode already diverged on the FIRST candle of its session day cannot be dated:
    # it may have been running since the overnight session and only looks like it started
    # at the window open. True drops those; set False to count them as onsets anyway.
    # Scalar or per-class dict, like div_abs_pp. True drops any episode whose run begins on a
    # discontinuity — the window's first bar, or a resume after a candle gap — because its birth
    # time is unknown. That is the DEFINITION of OPEN, but on INTRA it discards every spread that
    # was already wide at the window open, and those are disproportionately the big ones:
    # measured 2026-09-06, it dropped 52% of INTRA episodes and 73% of those peaking above 2pp,
    # while the convergence rate of the dropped ones was no worse (64.7% vs 63.8%).
    require_fresh_onset: object = True,
    session_split_min: int = 1020,
    # FEED ROLLOVER (measured on the published Scout logs, 2026-09-20): PairFlux PRE exits pile up at
    # exactly 00:00 (1183 vs 15 the minute before, 99.8% of them "wins") and again at 04:00 (208 vs ~10).
    # A market does not converge on the clock - the feed's Stack% baseline rolls over there, so a
    # deviation vanishes mechanically and the episode reads as resolved. Every clock time in
    # `rollover_minutes` (minutes on the PRE-wrapped axis: 00:00 = 0, 04:00 = 240) is therefore treated
    # as a HARD discontinuity: (1) no divergence or convergence run spans it, (2) an episode alive at it is
    # CUT there - it can only resolve inside its own segment, else it is a forced exit on the last row before
    # the rollover (its capture measured on the baseline it was entered on), and (3) a run that starts in
    # the `rollover_guard_min` minutes after it is trimmed to begin after the guard, or dropped if too short
    # (a deviation "born" on the first bars of a new baseline is the baseline, not the market). Classes whose
    # window never reaches these minutes (OPEN, INTRA) are untouched. () / 0 restores the old behaviour.
    rollover_minutes: tuple = (0, 240),
    rollover_guard_min: int = 10,
    # candle size; None = infer from the staged data (mode of the positive smin steps)
    bar_minutes: Optional[int] = None,
    # "ols"  -> dev = Stack%_A - (alpha + beta*Stack%_B), beta/alpha fitted per (pair, class)
    # "unit" -> dev = Stack%_A - Stack%_B, the plain "both should have moved the same %"
    hedge_mode: str = "ols",
    # episode thresholds, in z units of the pair's own spread (scale-free across pairs)
    div_z: Optional[float] = 2.0,
    conv_z: Optional[float] = 0.5,
    # Absolute thresholds in PERCENTAGE POINTS, ANDed with the z ones. z alone answers "is
    # this unusual for this pair", which is not the same question as "is this worth trading":
    # on a tight pair like AAAU/GLD a clean z=2.4 divergence measures 0.06pp. Set a side to
    # None to drop that condition; at least one divergence condition must remain.
    # Scalar = the same threshold for every class. Dict = one entry per class, which is how
    # INTRA runs at 1.0/0.3 while PRE and OPEN stay on the 0.5/0.2 they were measured at. A
    # dict must name every class; a silent fallback here would be a threshold nobody chose.
    div_abs_pp: Optional[object] = None,
    conv_abs_pp: Optional[object] = None,
    # How the z scale is estimated. "std" is the textbook z-score, but it has a trap: a pair
    # that spends a large slice of the window diverged inflates its own sigma, so the very
    # divergence you are hunting stops clearing div_z and the pair silently scores 0 episodes.
    # "mad" (median / 1.4826*MAD) takes the scale from the QUIET state instead, so long or
    # frequent divergences stay visible. Try "mad" first if a class comes back suspiciously empty.
    scale_mode: str = "std",
    # Where "dev == 0" sits. Mean/OLS centring puts zero at the pair's AVERAGE spread, which
    # drifts off the resting state whenever divergences are one-sided — and then an absolute
    # conv_abs_pp band around zero is unreachable no matter how the pair behaves. Median
    # centring puts zero at the state the pair actually spends most of its time in, which is
    # what an absolute threshold needs. "auto" = median as soon as anything depends on the
    # resting state (any *_abs_pp threshold, or scale_mode="mad").
    center_mode: str = "auto",       # "zero" | "mean" | "median" | "auto"
    # Economic floor: drop episodes whose peak deviation is below this many percentage points.
    # A spread can be statistically extreme and still be too small to trade.
    min_abs_peak_pp: float = 0.0,
    # Ceiling on the peak. A 60pp gap between two stocks' daily moves is single-name news or
    # a stale print, not a spread that was ever going to close — and it drags SIG up while
    # pushing RATE down. 0 = no ceiling.
    max_abs_peak_pp: float = 0.0,
    # SOFT RESOLUTION, exactly as ArbitRage scores an event. A divergence that never gets all the
    # way back inside conv_abs_pp but retraces to peak/soft_ratio after its peak has still come
    # back in the sense that matters, and ArbitRage has always counted it. PairFlux counted only
    # the HARD return, which is why its rates read far below ArbitRage's on the same kind of event.
    #
    # Set to None to score HARD only.
    soft_ratio: Optional[float] = 3.0,
    # NORMALISATION: both the divergence peak and the return-to-zero must survive this many
    # CONSECUTIVE candles. Single-candle spikes and single-candle touches of zero are noise
    # and must not create or resolve an episode.
    min_hold: int = 3,
    # "Consecutive" candles are decided on the CLOCK, not on row adjacency. Overnight and
    # pre-market bars are irregular (measured on real data: ~3 bars per ticker per overnight
    # session, median step 4 min), so demanding three strictly 1-minute-apart candles makes
    # an episode almost impossible to form there. A gap wider than this many minutes breaks
    # the run; None = require the exact inferred bar step (strict).
    max_gap_minutes: Optional[int] = None,
    # candidate filter (step 1 of the classic pair-trading checklist)
    min_corr: float = 0.7,
    # "Moves synchronously" means beta near 1. A 3x leveraged ETF against its own index is
    # geared, not synchronous: its spread is a mechanical function of the underlying move,
    # not a mispricing that has to revert. beta_band=1.5 keeps only pairs with beta inside
    # [1/1.5, 1.5]; None = no filter. Measured on the first pp-threshold run: 56% of the
    # top-200 INTRA pairs were geared-ETF relationships.
    beta_band: Optional[float] = None,
    # Correlation is measured on k-bar returns, not 1-bar. One-minute returns are mostly
    # microstructure noise, so 1-bar correlation between two ordinary stocks sits around
    # 0.2-0.4 and the 0.7-0.8 rule of thumb (which comes from DAILY data) would reject
    # everything. 5-bar returns are far more stable. If a class prints "no pair reaches
    # corr>=...", the log also prints the best corr actually seen — tune against that.
    corr_step_bars: int = 5,
    max_pairs_per_bench: int = 20000,
    corr_max_rows: int = 20000,          # subsample rows for the corr matmuls only
    # coverage guards
    min_bars_per_ticker: int = 500,
    min_days_per_ticker: int = 10,
    max_tickers_per_bench: int = 800,
    max_matrix_mb: int = 2000,
    # output filter
    min_total: int = 5,                  # keep a pair if ANY class reaches this many episodes
    # Quality gate, not just an evidence gate: a pair is kept only if SOME class reaches
    # min_total episodes AND converges at least this often. min_total alone lets a pair with
    # 40 episodes and a 12% rate into the CSV, which is noise you then have to filter by hand.
    # None = no rate gate.
    min_rate: Optional[float] = None,
    # The same gate on the SUCCESS rate instead of the convergence rate: win_rate counts
    # every episode that ended in profit, including ones that never converged but still
    # came back part of the way before the forced exit. None = no gate.
    min_win_rate: Optional[float] = None,
    # SIGMA — поріг на пару й клас: найбільше відхилення, з якого пара ще надійно
    # зводиться. Шукається вгору від sigma_min_pp кроком sigma_step_pp; рівень
    # зараховується, коли нижня межа Вілсона для P(зійшлася | пік >= рівень) досягає
    # sigma_min_rate при щонайменше sigma_min_total епізодах.
    # Where the sigma walk starts. Scalar, per-class dict, or None to follow div_abs_pp.
    # It is clamped up to div_abs_pp either way: episodes only exist at or above the entry
    # threshold, so a lower start would report a level nothing could be opened at.
    sigma_min_pp: object = None,
    # Optional EXTRA gate on the chosen level: None = off, the capture test decides alone.
    sigma_min_rate: Optional[float] = None,
    # GAMMA: where the search STARTS, per class (scalar or dict). Unlike sigma this floor is
    # not tied to div_abs_pp — it is set well above it on purpose, because expected capture
    # rises with the entry level without turning over, so the floor IS the quality dial.
    # None disables gamma entirely. Grid step and evidence bar below.
    gamma_floor_pp: object = None,
    gamma_step_pp: float = 1.0,
    gamma_min_total: int = 10,
    # Exit as a FRACTION of the deviation actually entered at. Measured better than any
    # absolute exit: 0.75 gave the highest win rate (62.7% marginal) and, crucially, the
    # best worst-block result, because it does not wait for a full convergence that a 12pp
    # spread often never delivers inside one session.
    gamma_exit_ratio: float = 0.75,
    # No NEW gamma entry after this clock time. At a 12pp floor this is worth roughly a
    # point per trade (+1.24 -> +2.11) because the spread needs session left to come back.
    gamma_cutoff_hm: tuple = (13, 0),
    # How hard the level has to prove itself. 0.0 = mean capture above zero; 1.96 = the 95%
    # lower bound above zero. Measured on INTRA at floor 4.0 over a 70/30 date split:
    #   z=0.00  518 pairs (3.5% of them)  +0.709 pp/trade  56.4% of trades profitable
    #   z=1.96   39 pairs (0.3%)          +1.910 pp/trade  66.7%
    # Coverage and edge trade off directly — relaxing the floor to 1.0 covers 63% of pairs
    # and turns the edge NEGATIVE (-0.033), so there is no setting that is both broad and good.
    gamma_z: float = 0.0,
    # Entries wider than this are dropped as data faults, matching the cap the floors
    # were measured under. Scalar or per-class dict.
    gamma_max_dev_pp: object = 25.0,
    sigma_step_pp: float = 0.1,
    sigma_min_total: int = 5,
    # Evidence bar for the RANKED list specifically: score = conv_per_day, so a pair
    # with 3 lucky cycles must not outrank one that produces them steadily.
    # on almost no evidence. None = same as min_total.
    best_min_total: Optional[int] = None,
    top_k_best: int = 500,
    # Augmented Dickey-Fuller on the spread. Off by default: it costs far more than every
    # other statistic combined and, because Stack% resets to 0 every session, the pooled
    # series it runs on is a concatenation of daily segments rather than one long process.
    # half_life / mr_lambda below are day-aware and answer the same practical question.
    compute_adf: bool = False,
    adf_maxlag: int = 1,
    log_every_n_pairs: int = 5000,
):
    """
    PairFlux: rate how reliably a pair of same-benchmark tickers CONVERGES after diverging.

    Deviation (the thing that diverges):
      Stack% is each ticker's % move against its own previous close, so two tickers that
      trade together "should" print the same Stack%, and both legs start every session at
      exactly 0. With center_mode="zero" (recommended) the spread is measured straight from
      that natural anchor: dev = Stack%_A - beta*Stack%_B, beta fitted through the origin,
      no intercept and no re-centring. Otherwise the deviation is what they actually do
      minus what the fitted model says they should:
          hedge_mode="ols"  dev = Stack%_A - (alpha_ols + beta * Stack%_B)
          hedge_mode="unit" dev = Stack%_A - Stack%_B - mean(Stack%_A - Stack%_B)
      alpha_ols/beta are fitted per (pair, class) — the relationship at 03:00 is not the
      relationship at 11:00, so one global beta would smear all three classes together.
      z = dev / std(dev) within the class.

      EVERY pair statistic is per (pair, class), not per pair: corr is measured on the class
      window's own return matrix, beta and alpha_ols are fitted on the class window's own rows,
      resid_std is that window's spread deviation, and alpha is the median converged peak inside
      it. A pair is therefore three separate objects — one per interval — and nothing about it is
      carried across class boundaries.

    Episode machine (per pair, per class, per session day):
      - DIVERGENCE: |z| >= div_z AND |dev| >= div_abs_pp (whichever of the two is set),
        held for >= min_hold consecutive candles.
      - PEAK: the largest |dev| that itself survived min_hold candles (a sliding minimum, so
        a one-candle spike can never set the peak).
      - CONVERGENCE: |z| <= conv_z AND |dev| <= conv_abs_pp (whichever is set), held for
        >= min_hold candles, after the divergence and inside the same day and class window.
      - A converged episode CLOSES the event. The next divergence after it opens a new one,
        so a pair can legitimately produce several episodes in one session.
      - Divergence runs that are not separated by a convergence belong to the SAME episode
        (peak = the max across them). Without this rule one unresolved divergence that
        oscillates around the threshold would be counted as a dozen separate episodes and
        inflate both TOTAL and the failure count.
      - An episode still open when the class window ends counts as a FAILURE (it is also
        exported as "unresolved" so the censored variant can be re-derived).
      - ONSET: if the class has an onset window (OPEN: 9:00-9:25), only episodes born inside
        it are rated; they may still converge anywhere up to the end of the class window.

    Per pair x class:
      total     — every divergence episode
      converged — the ones that came back
      rate      — converged / total
      rate_lb   — Wilson 95% lower bound on rate; USE THIS TO RANK, not rate. rate=1.0 out of
                  3 episodes is not better than rate=0.82 out of 200, and plain rate says it is.
      alpha     — MEDIAN peak deviation among the CONVERGED episodes, in percentage points: how
                  far this pair typically stretches before it comes back. Per pair AND per class —
                  every pair has its own, and it differs between PRE, OPEN and INTRA.
                  Median rather than RMS: peaks are bounded below by div_abs_pp and unbounded
                  above, so a single news-driven episode that happened to converge would dominate a
                  squared average and the statistic would describe the outlier, not the pair.
      sigma     — the BREAK-EVEN ENTRY LEVEL, in percentage points: the lowest deviation at
                  which opening the trade is reliably profitable BEFORE execution cost. Walks
                  up from sigma_min_pp and takes the FIRST level whose mean capture - entering
                  there, exiting where the episode exited - is above zero on its 95% lower
                  bound. Gross: subtract your own measured spread cost when you use it.
                  Fresh onsets only: a level an episode was already past when its window
                  opened was never choosable. Null means no level on this pair ever paid.
                  It was previously the largest deviation the pair still comes back
                  from: the highest level where P(converge | peak >= level) holds up, floored at
                  sigma_min_pp. Per pair AND per class. This is the unit a live reading should be
                  divided by — "the spread is at 1.4 sigma" means 1.4x the level this pair
                  reliably returns from, which is a different question from alpha (how far it
                  typically stretches). sigma_rate is the convergence rate at that level and
                  sigma_n how many episodes back it; None when even sigma_min_pp fails the bar.
      gamma     — the RARE-BUT-RELIABLE entry level, pp, and the only figure here built from
                  SIMULATED TRADES rather than from episode statistics: enter at the deviation
                  actually on the screen once it has held beyond the level, exit at
                  gamma_exit_ratio of it, force out on the day's last candle, and take no new
                  entry after gamma_cutoff_hm. The walk starts at gamma_floor_pp, well above
                  the event threshold, and the pair may only raise it. Most pairs never
                  qualify and get null — that is the point: a shortlist, not a rating.
                  gamma_n is the trades behind it, gamma_rate the share of them PROFITABLE
                  (not a convergence rate), gamma_cap their mean capture in pp, gross.
                  Every knob is per class: the three windows have different deviation scales
                  and different amounts of session left, so they do not share a setting.
                  gamma_z sets how hard the level must prove itself and so how many pairs get
                  one at all: at floor 4.0, z=0 gives 518 INTRA pairs at +0.71 pp/trade, z=1.96
                  gives 39 at +1.91. Both held across three different train/test splits.
      wins      — episodes that ended in profit (capture > 0), converged or not.
      win_rate  — wins / total: SUCCESSFUL to ALL. This is the rate that accounts for the
                  forced exits; `rate` above only measures how often the spread came all the
                  way back inside conv_abs_pp. win_rate_lb is its Wilson lower bound.
      cap_mean / cap_p50 / cap_p10 — what a trade actually BANKS, over EVERY episode: the
                  distance from the confirmed entry to the exit, where the exit is the
                  confirmed convergence or, failing that, the last candle of the day. Losing
                  episodes carry a negative capture, so cap_mean is the expected take per
                  episode rather than the average win. cap_p10 is the pessimistic end.
      cap_lb    — lower 95% bound on cap_mean (mean - 1.96*se); kept as its own column.
      cap_conv_mean — the old win-only average, kept for comparison.
      alpha_z   — alpha in z units (alpha / resid_std).
      Also: separate long/short stats (dev>0 vs dev<0 — a pair is often not symmetric),
      median_bars_to_conv, corr, beta, alpha_ols, resid_std, mr_lambda, half_life, beta_drift.

    Ranking: score = cap_lb — the expected realised take per episode in percentage points,
    discounted for sample size. Signed, so a pair that loses money ranks below zero.
    """
    import gc, json, time, math, gzip, heapq
    from collections import defaultdict
    import numpy as np
    import pandas as pd
    import pyarrow.parquet as pq
    from pathlib import Path

    if class_windows is None:
        class_windows = CLASS_WINDOWS_DEFAULT
    if hedge_mode not in ("ols", "unit"):
        raise ValueError("hedge_mode must be 'ols' or 'unit'")
    if min_hold < 1:
        raise ValueError("min_hold must be >= 1")
    ROLL = tuple(sorted({int(m) for m in (rollover_minutes or ())}))
    ROLL_GUARD = max(0, int(rollover_guard_min or 0))
    ROLL_STATS = {"runs_trimmed": 0, "runs_dropped": 0, "episodes_cut": 0, "episodes": 0}
    if div_z is None and div_abs_pp is None:
        raise ValueError("set at least one of div_z / div_abs_pp")
    if div_z is not None and conv_z is not None and conv_z >= div_z:
        raise ValueError(f"conv_z ({conv_z}) must be below div_z ({div_z})")
    if scale_mode not in ("std", "mad"):
        raise ValueError("scale_mode must be 'std' or 'mad'")
    if center_mode not in ("zero", "mean", "median", "auto"):
        raise ValueError("center_mode must be 'zero', 'mean', 'median' or 'auto'")
    center_median = center_mode == "median" or (
        center_mode == "auto" and (scale_mode == "mad" or
                                   div_abs_pp is not None or conv_abs_pp is not None))
    # NOTE: read off the raw arguments, so a dict counts as "an absolute threshold is in use"
    # for every class. Centring must not differ between classes or dev would not be comparable.

    best_total_min = min_total if best_min_total is None else int(best_min_total)
    CLASSES = list(class_windows.keys())

    def _by_class(value, name):
        """A scalar applies to every class; a dict gives each class its own. A dict that omits
        a class raises rather than defaulting — the whole point of the dict is that the reader
        can see which number each class ran at."""
        if not isinstance(value, dict):
            return {c: value for c in CLASSES}
        missing = [c for c in CLASSES if c not in value]
        if missing:
            raise ValueError(f"{name} is a dict but does not name {missing}; "
                             f"give every class a value or pass a single number")
        return {c: value[c] for c in CLASSES}

    DIV_PP = _by_class(div_abs_pp, "div_abs_pp")
    CONV_PP = _by_class(conv_abs_pp, "conv_abs_pp")
    SIGMA_PP = _by_class(sigma_min_pp, "sigma_min_pp")
    for c in CLASSES:
        _floor = DIV_PP[c] if DIV_PP[c] is not None else 0.0
        if SIGMA_PP[c] is None or SIGMA_PP[c] < _floor:
            SIGMA_PP[c] = _floor
    FRESH = _by_class(require_fresh_onset, "require_fresh_onset")
    GAMMA_FLOOR = _by_class(gamma_floor_pp, "gamma_floor_pp")
    # Scalars still apply to every class; a dict gives each its own. A tuple like (13, 0) is
    # not a dict, so a single cutoff for all classes keeps working unchanged.
    GAMMA_EXIT = _by_class(gamma_exit_ratio, "gamma_exit_ratio")
    GAMMA_CUT = _by_class(gamma_cutoff_hm, "gamma_cutoff_hm")
    GAMMA_Z = _by_class(gamma_z, "gamma_z")
    GAMMA_MINN = _by_class(gamma_min_total, "gamma_min_total")
    GAMMA_STEP = _by_class(gamma_step_pp, "gamma_step_pp")
    GAMMA_MAXDEV = _by_class(gamma_max_dev_pp, "gamma_max_dev_pp")
    for c in CLASSES:
        if DIV_PP[c] is None and div_z is None:
            raise ValueError(f"class {c} has neither div_z nor div_abs_pp")
        if DIV_PP[c] is not None and CONV_PP[c] is not None and CONV_PP[c] >= DIV_PP[c]:
            raise ValueError(f"[{c}] conv_abs_pp ({CONV_PP[c]}) must be below "
                             f"div_abs_pp ({DIV_PP[c]})")

    CLS_SMIN = {c: (_to_smin(a, session_split_min), _to_smin(b, session_split_min))
                for c, (a, b) in class_windows.items()}
    if onset_windows is None:
        onset_windows = {"OPEN": ((9, 0), (9, 25))}
    ONSET_SMIN = {}
    for c in CLASSES:
        w = onset_windows.get(c)
        ONSET_SMIN[c] = CLS_SMIN[c] if w is None else (_to_smin(w[0], session_split_min),
                                                       _to_smin(w[1], session_split_min))
        olo, ohi = ONSET_SMIN[c]
        clo, chi = CLS_SMIN[c]
        if olo < clo or ohi > chi or olo > ohi:
            raise ValueError(f"onset window for {c} ({olo}..{ohi}) must sit inside its "
                             f"class window ({clo}..{chi})")

    try:
        from statsmodels.tsa.stattools import adfuller as _adfuller
    except Exception:
        _adfuller = None

    # FAIL BEFORE TOUCHING ANY OUTPUT. The stage check used to sit ~220 lines below the point
    # where summary.csv is truncated and its header written, so a missing stage produced a
    # HEADER-ONLY file — which parses cleanly and reads as "this strategy has no pairs" rather than
    # as an error. That is what the scheduled run published on 2026-09-01 and again on 2026-09-02.
    _stage_probe = Path(stage_dir)
    if not _stage_probe.exists() or not any(_stage_probe.glob("*.parquet")):
        raise FileNotFoundError(
            f"no staged parquet files in {stage_dir} — run stage 1 first (RUN_STAGE1=True). "
            f"Nothing has been written, so any previously published output is still intact."
        )

    for p in (output_onefile_jsonl, output_summary_csv, output_best_pairs_jsonl, output_episodes_jsonl):
        Path(p).parent.mkdir(parents=True, exist_ok=True)

    # ATOMIC PUBLISH. Everything is written to `<path>.tmp` and renamed into place only once the
    # run finishes. summary.csv is truncated at the start and appended to per benchmark, while
    # run_orion_daily.py copies the whole signals/ tree on its own schedule with no completeness
    # check — so without this a push landing mid-run publishes a partial file. The publisher
    # already ignores *.tmp, so it keeps the previous complete set instead.
    _tmp_of = {}

    def _stage_path(p):
        t = str(p) + ".tmp"
        _tmp_of[str(p)] = t
        return t

    def _publish_staged():
        import os as _os
        for _final, _tmp in _tmp_of.items():
            if _os.path.exists(_tmp):
                _os.replace(_tmp, _final)   # atomic within a filesystem

    # Keep the real names for the closing log — the variables below hold the staged ones.
    _final_names = (output_onefile_jsonl, output_summary_csv,
                    output_best_pairs_jsonl, output_episodes_jsonl)
    output_summary_csv = _stage_path(output_summary_csv)
    output_onefile_jsonl = _stage_path(output_onefile_jsonl)
    output_best_pairs_jsonl = _stage_path(output_best_pairs_jsonl)
    output_episodes_jsonl = _stage_path(output_episodes_jsonl)

    def _open_gz(path, mode="wt"):
        # NB: the staged name is "<name>.jsonl.gz.tmp", so test the name with any .tmp suffix
        # removed — otherwise the gzip outputs would silently be written as plain text.
        probe = str(path).lower()
        if probe.endswith(".tmp"):
            probe = probe[:-4]
        if probe.endswith(".gz"):
            return gzip.open(path, mode, encoding="utf-8", newline="\n", compresslevel=6)
        return open(path, mode.replace("t", ""), encoding="utf-8", newline="\n")

    CLS_FIELDS = ("total", "converged", "unresolved", "hard", "soft", "rate", "rate_lb",
                  "wins", "win_rate", "win_rate_lb", "alpha", "alpha_z",
                  "sigma", "sigma_rate", "sigma_n",
                  "gamma", "gamma_rate", "gamma_n", "gamma_cap",
                  "avg_peak", "p90_peak", "median_bars",
                  "cap_mean", "cap_p50", "cap_p10", "cap_lb", "cap_conv_mean",
                  "conv_per_day", "total_per_day", "score",
                  "long_total", "long_rate", "long_alpha",
                  "short_total", "short_rate", "short_alpha",
                  "corr", "beta", "alpha_ols", "resid_std", "mr_lambda", "half_life",
                  "beta_drift", "adf_t", "adf_p", "adf_stationary_5pct", "n_bars", "n_days")
    summary_cols = ["ticker_a", "ticker_b", "bench"] + [f"{c}_{f}" for c in CLASSES for f in CLS_FIELDS]
    pd.DataFrame(columns=summary_cols).to_csv(output_summary_csv, index=False, mode="w")

    onefile_f  = _open_gz(output_onefile_jsonl, "wt")
    episodes_f = _open_gz(output_episodes_jsonl, "wt") if write_episodes else None

    # ── small numeric helpers ────────────────────────────────────────────────
    def _js(x):
        if x is None: return None
        if isinstance(x, (np.floating, float)):
            return None if (np.isnan(x) or np.isinf(x)) else round(float(x), 6)
        if isinstance(x, (np.integer, int)): return int(x)
        if isinstance(x, (np.bool_, bool)): return bool(x)
        return x

    def _dstr(v):
        v = int(v)
        return f"{v // 10000:04d}-{(v // 100) % 100:02d}-{v % 100:02d}"

    def _wilson_lb(k, n, z=1.96):
        # Lower bound of the Wilson score interval. Shrinks small samples towards 0 instead
        # of letting 3/3 = 1.0 outrank 180/200 = 0.9.
        if n <= 0: return None
        p = k / n
        d = 1.0 + z * z / n
        c = p + z * z / (2 * n)
        m = z * math.sqrt(max(p * (1 - p) / n + z * z / (4 * n * n), 0.0))
        return max(0.0, (c - m) / d)

    def _runs(mask, brk):
        """Maximal runs of True in `mask`, additionally cut wherever brk[i] marks a
        discontinuity before position i (new session day or a hole in the candles)."""
        n = mask.size
        if n == 0:
            return np.empty(0, np.int64), np.empty(0, np.int64)
        prev = np.empty(n, bool); prev[0] = False; prev[1:] = mask[:-1]
        nxt = np.empty(n, bool); nxt[-1] = False; nxt[:-1] = mask[1:]
        brk_next = np.empty(n, bool); brk_next[-1] = True; brk_next[:-1] = brk[1:]
        starts = np.flatnonzero(mask & (~prev | brk))
        ends = np.flatnonzero(mask & (~nxt | brk_next)) + 1
        return starts, ends

    def _sustain_min(x, w):
        """y[i] = min(x[i:i+w]) — the level that held for w candles ending at i+w-1."""
        if w <= 1:
            return x
        if x.size < w:
            return np.empty(0, x.dtype)
        out = x[:x.size - w + 1].copy()
        for k in range(1, w):
            np.minimum(out, x[k:x.size - w + 1 + k], out=out)
        return out

    def _ols(x, y):
        n = x.size
        if n < 3: return 0.0, 1.0
        mx = x.mean(); my = y.mean()
        vx = float(((x - mx) ** 2).sum())
        if vx <= 0: return float(my - mx), 1.0
        beta = float(((x - mx) * (y - my)).sum() / vx)
        return float(my - beta * mx), beta

    def _mr_stats(dev, brk):
        """Day-aware mean reversion: d_dev_t = a + lam*dev_{t-1}. half_life = -ln2/ln(1+lam).
        Pairs straddling a session break are dropped, otherwise the daily reset of Stack%
        would be read as a gigantic reversion."""
        n = dev.size
        if n < 30:
            return None, None
        use = ~brk[1:]
        lag = dev[:-1][use]
        d = (dev[1:] - dev[:-1])[use]
        if lag.size < 30:
            return None, None
        a, lam = _ols(lag, d)
        # phi is the AR(1) coefficient of the spread. lam in (-1, 0) is ordinary decay.
        # lam in (-2, -1) is stationary but OSCILLATING — the spread overshoots zero every
        # bar (bid-ask bounce does exactly this on minute data) — and its envelope still
        # decays, so the half-life comes from |phi|. log1p(lam) is undefined at lam <= -1,
        # so it can never be used directly here.
        phi = 1.0 + lam
        if lam >= 0 or abs(phi) >= 1.0:
            return _js(lam), None
        if phi == 0.0:
            return _js(lam), 0.0          # full reversion inside one bar
        hl = -math.log(2.0) / math.log(abs(phi))
        return _js(lam), _js(hl)

    # Large-sample Dickey-Fuller critical values, constant / no trend.
    ADF_CRIT = {"10%": -2.57, "5%": -2.86, "1%": -3.43}

    def _adf(dev, brk):
        """-> (t_stat, p_value, stationary_at_5pct).

        p_value is only filled when statsmodels is importable — deriving a MacKinnon p-value
        by hand would mean hard-coding response-surface coefficients, and a wrong p-value is
        worse than none. Without statsmodels you still get the t-stat and the verdict against
        the standard critical value (5% = -2.86), which is what the decision actually needs.
        `pip install statsmodels` if you want the exact p."""
        if not compute_adf or dev.size < 50:
            return None, None, None
        if _adfuller is not None:
            try:
                r = _adfuller(dev, maxlag=adf_maxlag, autolag=None)
                return _js(r[0]), _js(r[1]), bool(r[1] < 0.05)
            except Exception:
                return None, None, None
        # numpy fallback: plain Dickey-Fuller with a constant (no augmentation), day-aware
        use = ~brk[1:]
        lag = dev[:-1][use]
        d = (dev[1:] - dev[:-1])[use]
        if lag.size < 50:
            return None, None, None
        X = np.column_stack([np.ones(lag.size), lag])
        coef, res, *_ = np.linalg.lstsq(X, d, rcond=None)
        resid = d - X @ coef
        dof = lag.size - 2
        if dof <= 0:
            return None, None, None
        s2 = float(resid @ resid) / dof
        xtx_inv = np.linalg.inv(X.T @ X)
        se = math.sqrt(max(s2 * xtx_inv[1, 1], 1e-30))
        t = float(coef[1] / se)
        return _js(t), None, bool(t < ADF_CRIT["5%"])

    def _pairwise_corr(R):
        """Masked pairwise correlation of every column against every other, tolerating NaN
        holes without dropping whole rows. Five matmuls instead of N^2 python pairs."""
        W = np.isfinite(R).astype(np.float32)
        X = np.where(np.isfinite(R), R, 0.0).astype(np.float32)
        X2 = X * X
        n = W.T @ W
        sx = X.T @ W
        sy = W.T @ X
        sxx = X2.T @ W
        syy = W.T @ X2
        sxy = X.T @ X
        with np.errstate(invalid="ignore", divide="ignore"):
            cov = n * sxy - sx * sy
            vx = n * sxx - sx * sx
            vy = n * syy - sy * sy
            c = cov / np.sqrt(vx * vy)
        c[~np.isfinite(c)] = np.nan
        np.fill_diagonal(c, np.nan)
        c[n < 30] = np.nan
        return c

    def _gamma_trades(dev, sdv, smv, bv, L, exit_ratio, cut_smin, max_dev):
        """-> array of capture, one per trade, entering when |dev| has HELD at or beyond L
        for min_hold candles and exiting when it has held back inside exit_ratio*|entry| for
        min_hold candles, or on the day's last candle if it never does.

        Entry is taken at the deviation that was really there, not at L: when the spread is
        already wider than L at the window open, 'entering at L' is a price that never
        existed. That fiction is what made the earlier episode-based estimate too kind.
        """
        n = dev.size
        h = min_hold
        nb_ = n - h + 1
        if nb_ <= 0:
            return np.empty(0)
        lo = dev[:nb_].copy(); hi = dev[:nb_].copy()
        for k in range(1, h):
            np.minimum(lo, dev[k:k + nb_], out=lo)
            np.maximum(hi, dev[k:k + nb_], out=hi)
        bad = np.zeros(nb_, bool)
        for k in range(1, h):
            bad |= bv[k:k + nb_]
        ok = ~bad                      # a window straddling a break is not h candles
        chg = np.empty(n, bool); chg[0] = True
        chg[1:] = sdv[1:] != sdv[:-1]
        starts = np.flatnonzero(chg)
        day_end = np.empty(n, np.int64)
        for i, st in enumerate(starts):
            en = (starts[i + 1] - 1) if i + 1 < starts.size else n - 1
            day_end[st:en + 1] = en
        up = np.flatnonzero(ok & (lo >= L))
        dn = np.flatnonzero(ok & (hi <= -L))
        if up.size == 0 and dn.size == 0:
            return np.empty(0)
        cand = np.concatenate([up, dn])
        side = np.concatenate([np.ones(up.size, np.int8), -np.ones(dn.size, np.int8)])
        o = np.argsort(cand, kind="stable")
        cand, side = cand[o], side[o]
        caps = []
        last = -1
        for t in range(cand.size):
            i0 = int(cand[t]) + h - 1
            if i0 <= last or smv[i0] > cut_smin:
                continue
            s = int(side[t])
            e = float(dev[i0])
            # A deviation this wide is a broken Stack%, not a spread. Without this the
            # PRE run handed back GDC/NNBR at gamma 275pp with a mean capture of
            # +3580pp - the same artefact tickers that produced 16 770pp peaks.
            if abs(e) > max_dev:
                continue
            tgt = exit_ratio * abs(e)
            de = int(day_end[i0])
            w0, w1 = i0 + 1, de - h + 1
            x = de
            if w1 >= w0:
                seg = slice(w0, w1 + 1)
                hit = (np.flatnonzero(ok[seg] & (hi[seg] <= tgt)) if s > 0
                       else np.flatnonzero(ok[seg] & (lo[seg] >= -tgt)))
                if hit.size:
                    x = w0 + int(hit[0]) + h - 1
            # BOTH ends have to be sane, not just the entry. Capture is entry minus exit, so a
            # broken exit print alone produced HAO/RGTX at +176pp a trade off entries capped at
            # 25pp - enough, on a pair with eight trades, to pick its level by itself. Pooled the
            # effect is nothing (0.04% of PRE trades, 0.001% of INTRA), per pair it is decisive.
            # The trade still blocks re-entry until its exit; it is only left out of the numbers.
            if abs(float(dev[x])) <= max_dev:
                caps.append(s * (dev[i0] - dev[x]))
            last = x
        return np.asarray(caps, float)

    def _roll_flags(sdate, smv):
        """roll[i] is True when row i is the first row at/after a feed-rollover clock time within its
        session day (the row before it is earlier than the rollover). Empty when no rollover is set."""
        roll = np.zeros(smv.size, bool)
        if not ROLL or smv.size < 2:
            return roll
        same = sdate[1:] == sdate[:-1]
        for b in ROLL:
            roll[1:] |= same & (smv[:-1] < b) & (smv[1:] >= b)
        return roll

    def _episodes(dev, z, sdate, smv, brk, div_pp, conv_pp):
        """-> (ep_start, peak, converged, hard, soft, bars, bars_conv, direction, entry_dev,
        exit_dev, capture, ep_fresh, entry_minute, exit_minute). `converged` = hard | soft, the
        ArbitRage definition of a normalised event. Every episode gets an exit, resolved or not.
        entry_minute/exit_minute are smv at the confirmed entry/exit row - the clock time a Scout
        "by time" chart bins on, not to be confused with `date` (the session day)."""
        absz = np.abs(z)
        absd = np.abs(dev)
        # the rollover is a discontinuity like a new day or a candle hole: no run spans it
        roll = _roll_flags(sdate, smv)
        brk = brk | roll
        segchg = roll.copy()
        segchg[1:] |= sdate[1:] != sdate[:-1]
        seg = np.cumsum(segchg)               # session day x rollover segment; == the day when ROLL is empty
        dmask = (absz >= div_z) if div_z is not None else np.ones(absd.size, bool)
        if div_pp is not None:
            dmask = dmask & (absd >= div_pp)
        ds, de = _runs(dmask, brk)
        keep = (de - ds) >= min_hold
        ds, de = ds[keep], de[keep]
        if ds.size == 0:
            return None
        if ROLL and ROLL_GUARD > 0:
            ing = np.zeros(smv.size, bool)
            for b in ROLL:
                ing |= (smv >= b) & (smv < b + ROLL_GUARD)
            inside = ing[ds]
            if inside.any():
                out_rows = np.flatnonzero(~ing)
                if out_rows.size:
                    nxt = np.searchsorted(out_rows, ds[inside])
                    first_out = np.where(nxt < out_rows.size, out_rows[np.minimum(nxt, out_rows.size - 1)], de[inside])
                else:
                    first_out = de[inside]
                ds = ds.copy()
                ds[inside] = first_out
                ROLL_STATS["runs_trimmed"] += int(inside.sum())
                keep = (de - ds) >= min_hold
                ROLL_STATS["runs_dropped"] += int((~keep).sum())
                ds, de = ds[keep], de[keep]
                if ds.size == 0:
                    return None
        cmask = (absz <= conv_z) if conv_z is not None else np.ones(absd.size, bool)
        if conv_pp is not None:
            cmask = cmask & (absd <= conv_pp)
        cs, ce = _runs(cmask, brk)
        cs = cs[(ce - cs) >= min_hold]

        sm = _sustain_min(np.abs(dev), min_hold)
        if sm.size == 0:
            return None
        idx = np.empty(2 * ds.size, dtype=np.int64)
        idx[0::2] = np.minimum(ds, sm.size - 1)
        idx[1::2] = np.clip(de - min_hold + 1, 0, sm.size - 1)
        run_peak = np.maximum.reduceat(sm, idx)[0::2]

        # the convergence run that resolves each divergence run; equal values == same episode
        res = np.searchsorted(cs, de)
        sd = seg[ds]
        new = np.empty(ds.size, bool); new[0] = True
        new[1:] = (res[1:] != res[:-1]) | (sd[1:] != sd[:-1])
        g = np.flatnonzero(new)

        ep_start = ds[g]
        ep_peak = np.maximum.reduceat(run_peak, g)
        ep_res = res[g]
        ok = ep_res < cs.size
        conv_pos = np.where(ok, cs[np.clip(ep_res, 0, max(cs.size - 1, 0))] if cs.size else 0, -1)
        converged = ok & (conv_pos >= 0)
        if cs.size:
            # resolved INSIDE its own segment: a convergence on the far side of a rollover is the baseline
            converged &= seg[np.clip(conv_pos, 0, seg.size - 1)] == seg[ep_start]
        hard = converged.copy()
        direction = np.sign(dev[ep_start])
        # What the trade actually banks. You do not enter at the peak: you enter when the
        # divergence is CONFIRMED (min_hold candles past the threshold), and you leave EITHER
        # when the convergence is confirmed, OR — if it never converges — at the LAST candle
        # of that session day inside the class window, at whatever the spread actually is.
        #
        # Recording that forced exit is what makes capture mean anything. Defined only on the
        # converged episodes it could never be bad: convergence means |dev| <= conv_abs_pp
        # after entering at div_abs_pp, so the take had an arithmetic floor of
        # (div_abs_pp - conv_abs_pp) and was never negative. Every statistic built on it was
        # therefore counting the wins and silently discarding the losses.
        #
        # capture is signed so that moving toward zero is positive: an overshoot past zero
        # counts as extra, and a spread that widened instead gives a NEGATIVE capture.
        n_dev = dev.size
        # last row index of each session day, then the one that closes each episode's own day
        day_end = np.flatnonzero(np.append(seg[1:] != seg[:-1], True))
        forced_exit = day_end[np.searchsorted(day_end, ep_start)]

        # SOFT: for an episode that never reached conv_abs_pp, the first bar AFTER its peak whose
        # level held at or below peak/soft_ratio for min_hold candles, inside the same day.
        #
        # Done as a loop over the UNRESOLVED episodes only, not over bars: the peak's POSITION is
        # needed and `np.maximum.reduceat` gives the value, not the argument. Each iteration scans
        # one episode's own slice, so the cost is bounded by the episode count, not the tape.
        soft = np.zeros(ep_start.size, bool)
        soft_pos = np.full(ep_start.size, -1, np.int64)
        if soft_ratio is not None and float(soft_ratio) > 0:
            _sr = float(soft_ratio)
            _last_sm = sm.size - 1
            for _i in np.flatnonzero(~hard):
                _a = int(ep_start[_i])
                # `sm` looks min_hold rows AHEAD, so on the last rows of a segment it reads the next segment
                # (the rows after a rollover: deviation ~0) and a held spread looks like it collapsed. With a
                # rollover set, a soft resolution must be confirmed by rows INSIDE the episode's own segment.
                _b = int(min(forced_exit[_i] - ((min_hold - 1) if ROLL else 0), _last_sm))
                if _b <= _a:
                    continue
                _seg = sm[_a:_b + 1]
                _pk_rel = int(np.argmax(_seg))
                _thr = float(ep_peak[_i]) / _sr
                _after = _seg[_pk_rel:]
                _hit = np.flatnonzero(_after <= _thr)
                if _hit.size:
                    soft[_i] = True
                    soft_pos[_i] = _a + _pk_rel + int(_hit[0])

        # An episode now RESOLVES either way, which is the ArbitRage definition of a normalised
        # event. `hard` and `soft` stay separate so the split is still reportable.
        converged = hard | soft
        bars_conv = np.where(hard, conv_pos - ep_start,
                             np.where(soft, soft_pos - ep_start, -1))

        entry_idx = np.minimum(ep_start + min_hold - 1, forced_exit)
        exit_idx = np.where(
            hard,
            np.minimum(np.clip(conv_pos + min_hold - 1, 0, n_dev - 1), forced_exit),
            np.where(soft,
                     np.minimum(np.clip(soft_pos + min_hold - 1, 0, n_dev - 1), forced_exit),
                     forced_exit))
        entry_dev = dev[entry_idx]
        exit_dev = dev[exit_idx]
        capture = np.sign(entry_dev) * (entry_dev - exit_dev)
        entry_minute = smv[entry_idx]
        exit_minute = smv[exit_idx]
        bars = exit_idx - ep_start          # actual holding period, every episode
        # An episode whose run begins on a discontinuity was ALREADY past its threshold when
        # the window opened. It counts for convergence statistics, but no entry level below
        # its opening width was ever choosable, so the break-even search must skip it.
        ep_fresh = ~brk[ep_start]
        if ROLL:
            ROLL_STATS["episodes"] += int(ep_start.size)
            _nx = np.minimum(forced_exit + 1, n_dev - 1)
            ROLL_STATS["episodes_cut"] += int((~converged & (forced_exit + 1 < n_dev) & roll[_nx]
                                               & (sdate[_nx] == sdate[forced_exit])).sum())
        return (ep_start, ep_peak, converged, hard, soft, bars, bars_conv, direction,
                entry_dev, exit_dev, capture, ep_fresh, entry_minute, exit_minute)

    # ── per-benchmark scan ───────────────────────────────────────────────────
    stage = Path(stage_dir)
    files = sorted(stage.glob("*.parquet"))
    if not files:
        raise FileNotFoundError(f"no staged parquet files in {stage_dir} — run stage 1 first")

    best_heaps = {c: [] for c in CLASSES}
    t0 = time.time()
    pairs_written = 0

    print(f"START PairFlux stage2  benches={len(files)}  hedge={hedge_mode}  "
          f"div_z={div_z} conv_z={conv_z} min_hold={min_hold}  min_corr={min_corr}")

    for fp in files:
        bench = fp.stem
        tb0 = time.time()
        df = pq.read_table(fp).to_pandas()
        if df.empty:
            continue

        tickers, tk_code = np.unique(df["ticker"].to_numpy(), return_inverse=True)
        # coverage guard before anything expensive
        cov = np.bincount(tk_code, minlength=tickers.size)
        ndays = pd.Series(df["sdate"].to_numpy()).groupby(tk_code).nunique().reindex(
            range(tickers.size)).fillna(0).to_numpy()
        good = (cov >= min_bars_per_ticker) & (ndays >= min_days_per_ticker)
        n_cov = int(good.sum())
        if n_cov < tickers.size:
            print(f"  [{bench}] {tickers.size - n_cov} of {tickers.size} tickers dropped by "
                  f"coverage (min_bars={min_bars_per_ticker}, min_days={min_days_per_ticker})")
        if n_cov > max_tickers_per_bench:
            # rank WITHIN the eligible set, and say so — this is a real narrowing of
            # "check every ticker" and must never happen silently
            elig = np.flatnonzero(good)
            keep = elig[np.argsort(-cov[elig])[:max_tickers_per_bench]]
            good[:] = False
            good[keep] = True
            print(f"  [{bench}] CAPPED to the {max_tickers_per_bench} best-covered tickers of "
                  f"{n_cov} eligible — raise max_tickers_per_bench to widen the scan")
        if good.sum() < 2:
            print(f"  [{bench}] skipped — only {int(good.sum())} tickers pass coverage")
            continue

        sel = np.flatnonzero(good)
        remap = -np.ones(tickers.size, np.int64)
        remap[sel] = np.arange(sel.size)
        keep_rows = remap[tk_code] >= 0
        col_of_row = remap[tk_code[keep_rows]]
        sdate_all = df["sdate"].to_numpy()[keep_rows]
        smin_all = df["smin"].to_numpy().astype(np.int32)[keep_rows]
        stack_all = df["stack"].to_numpy()[keep_rows]
        names = tickers[sel]
        del df
        gc.collect()

        if bar_minutes is None:
            s = np.sort(np.unique(smin_all))
            d = np.diff(s)
            d = d[d > 0]
            step = int(np.bincount(d).argmax()) if d.size else 1
        else:
            step = int(bar_minutes)
        gap_tol = step if max_gap_minutes is None else max(step, int(max_gap_minutes))

        pair_stats = defaultdict(dict)

        for cls in CLASSES:
            lo, hi = CLS_SMIN[cls]
            m = (smin_all >= lo) & (smin_all <= hi)
            if m.sum() < min_bars_per_ticker:
                continue
            sd_c = sdate_all[m]; sm_c = smin_all[m]
            col_c = col_of_row[m]; val_c = stack_all[m]

            row_key = sd_c.astype(np.int64) * 100000 + (sm_c.astype(np.int64) + 1440)
            uniq_rows, row_idx = np.unique(row_key, return_inverse=True)
            T, N = uniq_rows.size, names.size
            mb = T * N * 4 / 1e6
            if mb > max_matrix_mb:
                print(f"  [{bench}/{cls}] SKIPPED — matrix would be {mb:,.0f} MB "
                      f"({T:,} rows x {N} tickers). Narrow start_date or max_tickers_per_bench.")
                continue

            M = np.full((T, N), np.nan, dtype=np.float32)
            M[row_idx, col_c] = val_c
            r_sdate = (uniq_rows // 100000).astype(np.int32)
            r_smin = (uniq_rows % 100000 - 1440).astype(np.int32)
            brk = np.empty(T, bool); brk[0] = True
            brk[1:] = (r_sdate[1:] != r_sdate[:-1]) | (r_smin[1:] - r_smin[:-1] > gap_tol)

            # candidate filter on RETURNS, not on Stack% levels: two tickers both drifting up
            # all session correlate ~1 on levels no matter how they got there.
            kbar = max(1, int(corr_step_bars))
            if T <= kbar:
                del M
                gc.collect()
                continue
            cbrk = np.cumsum(brk.astype(np.int32))
            R = M[kbar:] - M[:-kbar]
            # a k-bar return is only valid if no session break or candle gap falls inside it
            R[(cbrk[kbar:] - cbrk[:-kbar]) > 0] = np.nan
            Rc = R
            if R.shape[0] > corr_max_rows:
                Rc = R[np.linspace(0, R.shape[0] - 1, corr_max_rows).astype(np.int64)]
            C = _pairwise_corr(Rc)
            iu = np.triu_indices(N, k=1)
            cvals = C[iu]
            cand = np.flatnonzero(np.isfinite(cvals) & (cvals >= min_corr))
            if cand.size == 0:
                print(f"  [{bench}/{cls}] no pair reaches corr>={min_corr} "
                      f"(best={np.nanmax(cvals) if np.isfinite(cvals).any() else float('nan'):.3f})")
                del M, R, C
                gc.collect()
                continue
            if cand.size > max_pairs_per_bench:
                cand = cand[np.argsort(-cvals[cand])[:max_pairs_per_bench]]
            ai, bi = iu[0][cand], iu[1][cand]
            print(f"  [{bench}/{cls}] rows={T:,} tickers={N} pairs={cand.size:,} "
                  f"({mb:,.0f} MB matrix, step={step}m)")

            for k in range(cand.size):
                ia, ib = int(ai[k]), int(bi[k])
                a = M[:, ia]; b = M[:, ib]
                v = np.isfinite(a) & np.isfinite(b)
                if v.sum() < min_bars_per_ticker:
                    continue
                va = a[v].astype(np.float64); vb = b[v].astype(np.float64)
                sdv = r_sdate[v]
                # recompute breaks on the pair's own valid grid: a hole in EITHER leg breaks
                # the run, otherwise "3 consecutive candles" would silently span a gap
                smv = r_smin[v]
                bv = np.empty(va.size, bool); bv[0] = True
                bv[1:] = (sdv[1:] != sdv[:-1]) | (smv[1:] - smv[:-1] > gap_tol)

                if center_mode == "zero":
                    # Stack% is each ticker's move against its OWN previous close, so both
                    # legs start every session at exactly 0. The spread therefore has a real
                    # anchor at zero and must not be re-centred: beta is fitted THROUGH THE
                    # ORIGIN and alpha is pinned to 0, making dev literally A - beta*B. A
                    # fitted intercept would move "no deviation" off true parity, and a
                    # persistent one-sided drift would then be silently absorbed into it.
                    if hedge_mode == "ols":
                        den = float(vb @ vb)
                        beta = float((va @ vb) / den) if den > 0 else 1.0
                    else:
                        beta = 1.0
                    alpha_ols = 0.0
                elif hedge_mode == "ols":
                    alpha_ols, beta = _ols(vb, va)
                else:
                    # beta pinned to 1, but alpha still centres the spread so that "dev == 0"
                    # means the same thing in both modes: the pair sits at its own equilibrium
                    beta = 1.0
                    alpha_ols = float((va - vb).mean())
                if beta_band is not None and not (1.0 / beta_band <= beta <= beta_band):
                    continue
                dev = va - (alpha_ols + beta * vb)
                if center_median:
                    med = float(np.median(dev))
                    dev = dev - med
                    alpha_ols += med
                if scale_mode == "mad":
                    sc = float(np.median(np.abs(dev - np.median(dev)))) * 1.4826
                    # MAD collapses to 0 on a spread that is flat more than half the time
                    sd_dev = sc if sc > 1e-9 else float(dev.std())
                else:
                    sd_dev = float(dev.std())
                if not np.isfinite(sd_dev) or sd_dev <= 1e-9:
                    continue
                z = dev / sd_dev

                ep = _episodes(dev, z, sdv, smv, bv, DIV_PP[cls], CONV_PP[cls])
                if ep is None:
                    continue
                (ep_start, peak, conv, hard, soft, bars, bars_conv, dirn,
                 entry_dev, exit_dev, capture, ep_fresh, entry_minute, exit_minute) = ep
                olo, ohi = ONSET_SMIN[cls]
                _fresh = FRESH[cls]
                if (olo, ohi) != (lo, hi) or _fresh:
                    m = (smv[ep_start] >= olo) & (smv[ep_start] <= ohi)
                    if _fresh:
                        # a run beginning exactly on a discontinuity (day start or a hole in
                        # the candles) has an unknown birth time — it is not an onset
                        m &= ~bv[ep_start]
                    if not m.any():
                        continue
                    (ep_start, peak, conv, hard, soft, bars, bars_conv, dirn,
                     entry_dev, exit_dev, capture, ep_fresh, entry_minute, exit_minute) = (
                        ep_start[m], peak[m], conv[m], hard[m], soft[m], bars[m], bars_conv[m],
                        dirn[m], entry_dev[m], exit_dev[m], capture[m], ep_fresh[m],
                        entry_minute[m], exit_minute[m])
                if min_abs_peak_pp > 0 or max_abs_peak_pp > 0:
                    m = np.ones(peak.size, bool)
                    if min_abs_peak_pp > 0:
                        m &= peak >= min_abs_peak_pp
                    if max_abs_peak_pp > 0:
                        m &= peak <= max_abs_peak_pp
                    if not m.any():
                        continue
                    (ep_start, peak, conv, hard, soft, bars, bars_conv, dirn,
                     entry_dev, exit_dev, capture, ep_fresh, entry_minute, exit_minute) = (
                        ep_start[m], peak[m], conv[m], hard[m], soft[m], bars[m], bars_conv[m],
                        dirn[m], entry_dev[m], exit_dev[m], capture[m], ep_fresh[m],
                        entry_minute[m], exit_minute[m])
                total = int(ep_start.size)
                nconv = int(conv.sum())
                nhard = int(hard.sum())
                nsoft = int(soft.sum())
                pk_c = peak[conv]
                # SIGMA = the MEDIAN peak deviation among the episodes that CAME BACK. Per pair, per
                # class. Read it as "how far this pair typically stretches before converging", which
                # is the number a live reading should be compared against.
                #
                # Median, not RMS: RMS squares before averaging, so one 12pp news-driven episode
                # that happened to converge drags the figure far above anything the pair normally
                # does, and the statistic then describes the outlier rather than the pair. Peaks are
                # right-skewed by construction (bounded below by div_abs_pp, unbounded above), which
                # is exactly the shape where a mean-like estimator misleads.
                alpha = float(np.median(pk_c)) if pk_c.size else None
                # SIGMA — найбільше відхилення, з якого ця пара ЩЕ повертається.
                # Йдемо вгору від sigma_min_pp і питаємо: серед епізодів, що дотягнули
                # принаймні до цього рівня, яка частка зійшлася? Рівень зараховується за
                # нижньою межею Вілсона, а не за сирою часткою — інакше 3/3 обходить 180/200,
                # чого цей файл уникає скрізь. None, якщо навіть sigma_min_pp не проходить.
                # SIGMA — беззбитковий рівень входу: найнижче відхилення, з якого відкриття
                # позиції надійно варте більше, ніж коштує.
                #
                # Ідемо вгору від sigma_min_pp і питаємо: якби ми відкривались рівно на
                # цьому рівні щоразу, коли пара його досягала, скільки б взяли?
                # Забираємо   L - sign(entry_dev) * exit_dev,   бо вихід належить епізоду
                # (зведення або кінець дня) і не залежить від точки входу.
                # Перший рівень, де НИЖНЯ межа середнього вища за нуль, і є сігмою.
                #
                # Це ВАЛОВИЙ рівень — витрати на виконання сюди НЕ входять. Їх видно на
                # живих даних (різниця між мідом і виконуваною ціною), і зашита сюди
                # заглушка лише ховала б їх усередині опублікованого числа.
                #
                # Найнижчий, а не найкращий: очікуваний заробіток росте з рівнем без
                # межі (виграш масштабується з L, а вихід зафіксований на conv_abs_pp), тож
                # максимуму не існує — існує лише поріг, нижче якого входити не варто.
                #
                # Досягнення рівня — це max(peak, |entry_dev|): peak є ВИТРИМАНИМ максимумом
                # і в 13% епізодів нижчий за миттєвий друк на вході.
                sigma = sigma_rate = None
                sigma_n = 0
                if total and ep_fresh.any():
                    _reach = np.maximum(peak[ep_fresh], np.abs(entry_dev[ep_fresh]))
                    _sx = (np.sign(entry_dev) * exit_dev)[ep_fresh]
                    _cv = conv[ep_fresh]
                    lvl, top = float(SIGMA_PP[cls]), float(_reach.max())
                    while lvl <= top + 1e-9:
                        at = _reach >= lvl - 1e-9
                        n_at = int(at.sum())
                        if n_at < sigma_min_total:
                            break
                        cap_at = lvl - _sx[at]
                        if n_at > 1:
                            se = float(cap_at.std(ddof=1)) / math.sqrt(n_at)
                            lb = float(cap_at.mean()) - 1.96 * se
                        else:
                            lb = -1.0
                        r_at = float(_cv[at].mean())
                        if lb > 0 and (sigma_min_rate is None or r_at >= sigma_min_rate):
                            sigma, sigma_n, sigma_rate = lvl, n_at, r_at
                            break
                        lvl = round(lvl + sigma_step_pp, 6)
                # GAMMA — рівень входу для РІДКІСНИХ АЛЕ НАДІЙНИХ ситуацій.
                #
                # Ідемо вгору від gamma_floor_pp і беремо НАЙНИЖЧИЙ рівень, на якому
                # СИМУЛЬОВАНІ трейди цієї пари дали додатний середній з довірою
                # (mean - gamma_z * se > 0) при щонайменше gamma_min_total трейдах.
                #
                # Це НЕ статистика епізодів: _gamma_trades програє саму угоду — вхід за
                # фактичним відхиленням, вихід на частку від нього, примусове закриття в
                # кінці дня. Попередня версія оцінювала вхід рівно на L і тим завищувала
                # результат майже втричі.
                #
                # Заміряно 2026-09-10, INTRA, три послідовні блоки сесій, кожного разу
                # перенавчаючи гамму на всіх попередніх (бета й центрування — на першій
                # половині, тож жоден блок їх не бачив):
                #   14-24 пари, 3.7 трейди/день, 71.9% виграшів, +2.11 pp/трейд,
                #   нижня межа +1.40, кожен з трьох блоків додатний окремо.
                # Підлога нижче дає більше трейдів і менше на трейд: 8pp — 49/день при
                # +0.71, 10pp — 20/день при +0.73. Вище 14pp все розвалюється (замало даних).
                gamma = gamma_rate = gamma_cap = None
                gamma_n = 0
                _gf = GAMMA_FLOOR[cls]
                if _gf is not None:
                    _cut = _to_smin(GAMMA_CUT[cls], session_split_min)
                    _exr, _gz = GAMMA_EXIT[cls], GAMMA_Z[cls]
                    _gmn, _gstep = GAMMA_MINN[cls], GAMMA_STEP[cls]
                    _gmax = GAMMA_MAXDEV[cls]
                    _top = float(np.abs(dev).max())
                    _best = None
                    lvl = float(_gf)
                    while lvl <= _top + 1e-9:
                        _tc = _gamma_trades(dev, sdv, smv, bv, lvl, _exr, _cut, _gmax)
                        # trade count falls as the level rises, so once there is too little
                        # evidence here there is less above: stop rather than keep walking
                        if _tc.size < _gmn:
                            break
                        _m = float(_tc.mean())
                        _se = float(_tc.std(ddof=1)) / math.sqrt(_tc.size)
                        if _m - _gz * _se > 0:
                            # rank the qualifying levels by what they BANKED IN TOTAL, so a
                            # pair whose edge is really at 15pp is not pinned to the floor.
                            # Measured 2026-09-10 against the alternatives, out of sample over
                            # three blocks: total 76.6% of trades profitable at +2.085 pp each,
                            # vs 70.7% / +1.901 for taking the lowest qualifying level and
                            # 72.0% / +1.880 for a plain argmax of the mean, which overfits.
                            # It also makes the value genuinely per-pair: 46% sit on the floor
                            # instead of 89%.
                            _score = _m * _tc.size
                            if _best is None or _score > _best[0]:
                                _best = (_score, lvl, int(_tc.size), _m,
                                         float((_tc > 0).mean()))
                        lvl = round(lvl + _gstep, 6)
                    if _best is not None:
                        _, gamma, gamma_n, gamma_cap, gamma_rate = _best
                rate = nconv / total if total else None
                rate_lb = _wilson_lb(nconv, total)
                # capture now exists for EVERY episode, so these describe the whole
                # distribution — the losses from the unresolved ones included — rather than
                # only the subset that happened to work.
                cap_all = capture[np.isfinite(capture)]
                nwin = int((cap_all > 0).sum())
                win_rate = nwin / cap_all.size if cap_all.size else None
                win_rate_lb = _wilson_lb(nwin, cap_all.size)
                cap_mean = float(cap_all.mean()) if cap_all.size else None
                cap_p50 = float(np.median(cap_all)) if cap_all.size else None
                # the pessimistic end: 1 episode in 10 gives you no more than this
                cap_p10 = float(np.percentile(cap_all, 10)) if cap_all.size else None
                # Lower 95% bound on the MEAN take (mean - 1.96*se). cap_mean alone lets a pair
                # with 3 episodes outrank one with 60 at the same average; this does not.
                if cap_all.size >= 2:
                    cap_lb = cap_mean - 1.96 * float(cap_all.std(ddof=1)) / math.sqrt(cap_all.size)
                else:
                    cap_lb = None
                # the old win-only number, kept so the two can be compared directly
                cap_c = capture[conv]; cap_c = cap_c[np.isfinite(cap_c)]
                cap_conv_mean = float(cap_c.mean()) if cap_c.size else None

                def _dir_stats(sign):
                    dm = dirn == sign
                    tt = int(dm.sum())
                    if tt == 0: return 0, None, None
                    cc = conv & dm
                    pk = peak[cc]
                    # Same median definition as `alpha`, restricted to one direction of the spread.
                    return (tt, round(int(cc.sum()) / tt, 4),
                            _js(float(np.median(pk)) if pk.size else None))

                lt, lr, ls = _dir_stats(1.0)
                st_, sr, ss = _dir_stats(-1.0)

                nd = int(np.unique(sdv).size)
                lam, hl = _mr_stats(dev, bv)
                adf_t, adf_p, adf_s5 = _adf(dev, bv)
                # split-half beta: a pair whose hedge ratio drifts is not the same pair any more
                half = va.size // 2
                if hedge_mode == "ols" and half > 30:
                    _, b1 = _ols(vb[:half], va[:half])
                    _, b2 = _ols(vb[half:], va[half:])
                    bdrift = abs(b2 - b1)
                else:
                    bdrift = None

                key = (str(names[ia]), str(names[ib]))
                pair_stats[key][cls] = {
                    "total": total, "converged": nconv, "unresolved": total - nconv,
                    # the ArbitRage split: hard = back inside conv_abs_pp, soft = back to
                    # peak/soft_ratio after the peak. rate counts both.
                    "hard": nhard, "soft": nsoft,
                    "rate": _js(rate), "rate_lb": _js(rate_lb),
                    "alpha": _js(alpha), "alpha_z": _js(alpha / sd_dev if alpha is not None else None),
                    "sigma": _js(sigma), "sigma_rate": _js(sigma_rate), "sigma_n": sigma_n,
                    "gamma": _js(gamma), "gamma_rate": _js(gamma_rate),
                    "gamma_n": gamma_n, "gamma_cap": _js(gamma_cap),
                    "avg_peak": _js(float(pk_c.mean()) if pk_c.size else None),
                    "p90_peak": _js(float(np.percentile(pk_c, 90)) if pk_c.size else None),
                    "median_bars": _js(float(np.median(bars_conv[conv])) if nconv else None),
                    "wins": nwin, "win_rate": _js(win_rate), "win_rate_lb": _js(win_rate_lb),
                    "cap_mean": _js(cap_mean), "cap_p50": _js(cap_p50), "cap_p10": _js(cap_p10),
                    "cap_lb": _js(cap_lb), "cap_conv_mean": _js(cap_conv_mean),
                    "conv_per_day": _js(nconv / nd if nd else None),
                    "total_per_day": _js(total / nd if nd else None),
                    # Ranked on HOW OFTEN the pair completes a diverge->converge cycle, which is
                    # what the strategy is looking for: a pair that is worth watching has to
                    # produce setups, not merely have a good hit rate on three of them. min_rate
                    # is the reliability half of the question and gates entry to the file at all;
                    # this is the frequency half. cap_lb stays its own column so the economics
                    # are visible, but it must not drive the list — ranking by it puts pairs of
                    # near-identical instruments (two 2x ETFs on the same underlying) on top,
                    # because their spread is mechanically tight rather than genuinely mean
                    # reverting.
                    "score": _js(nconv / nd if nd else None),
                    "long_total": lt, "long_rate": lr, "long_alpha": ls,
                    "short_total": st_, "short_rate": sr, "short_alpha": ss,
                    "corr": _js(float(cvals[cand[k]])),
                    "beta": _js(beta), "alpha_ols": _js(alpha_ols), "resid_std": _js(sd_dev),
                    "mr_lambda": lam, "half_life": hl, "beta_drift": _js(bdrift),
                    "adf_t": adf_t, "adf_p": adf_p, "adf_stationary_5pct": adf_s5,
                    "n_bars": int(va.size), "n_days": nd,
                }

                if write_episodes:
                    a_n, b_n = key
                    for j in range(total):
                        episodes_f.write(json.dumps({
                            "a": a_n, "b": b_n, "bench": bench, "cls": cls,
                            "date": _dstr(sdv[ep_start[j]]),
                            "peak": _js(float(peak[j])),
                            "peak_z": _js(float(peak[j] / sd_dev)),
                            "entry_dev": _js(float(entry_dev[j])),
                            "exit_dev": _js(float(exit_dev[j])),
                            "entry_minute": int(entry_minute[j]),
                            "exit_minute": int(exit_minute[j]),
                            "capture": _js(float(capture[j])) if np.isfinite(capture[j]) else None,
                            "converged": bool(conv[j]),
                            "kind": "hard" if hard[j] else ("soft" if soft[j] else "none"),
                            "bars": int(bars[j]),
                            "dir": int(dirn[j]),
                        }, ensure_ascii=False) + "\n")

                if log_every_n_pairs and (k + 1) % log_every_n_pairs == 0:
                    print(f"    ...{k+1:,}/{cand.size:,} pairs  elapsed={time.time()-tb0:.1f}s")

            del M, R, C
            gc.collect()

        # ── emit this benchmark's pairs ──
        rows = []
        for (a_n, b_n), per_cls in pair_stats.items():
            def _cls_ok(c):
                dd = per_cls.get(c) or {}
                if dd.get("total", 0) < min_total:
                    return False
                # rate/win_rate are None only when total == 0, already rejected above
                if min_rate is not None and (dd.get("rate") or 0.0) < min_rate:
                    return False
                return min_win_rate is None or (dd.get("win_rate") or 0.0) >= min_win_rate
            if not any(_cls_ok(c) for c in CLASSES):
                continue
            onefile_f.write(json.dumps({
                "a": a_n, "b": b_n, "bench": bench,
                "params": {
                    "hedge_mode": hedge_mode, "div_z": div_z, "conv_z": conv_z,
                    "min_hold": min_hold, "min_corr": min_corr, "soft_ratio": soft_ratio,
                    "min_total": min_total, "min_rate": min_rate,
                    "min_win_rate": min_win_rate,
                    "best_min_total": best_total_min,
                    "class_windows": {c: [list(x) for x in class_windows[c]] for c in CLASSES},
                    "onset_smin": {c: list(ONSET_SMIN[c]) for c in CLASSES},
                    "require_fresh_onset": FRESH,
                    "div_z": div_z, "conv_z": conv_z,
                    "scale_mode": scale_mode, "center_median": center_median,
                    "div_abs_pp": DIV_PP, "conv_abs_pp": CONV_PP, "sigma_min_pp": SIGMA_PP,
                    "max_gap_minutes": max_gap_minutes, "gap_tol": gap_tol,
                    "min_abs_peak_pp": min_abs_peak_pp, "max_abs_peak_pp": max_abs_peak_pp,
                    "session_split_min": session_split_min, "bar_minutes": step,
                    "rollover_minutes": list(ROLL), "rollover_guard_min": ROLL_GUARD,
                },
                "classes": per_cls,
            }, ensure_ascii=False) + "\n")
            row = {"ticker_a": a_n, "ticker_b": b_n, "bench": bench}
            for c in CLASSES:
                d = per_cls.get(c) or {}
                for f in CLS_FIELDS:
                    row[f"{c}_{f}"] = d.get(f)
                # no "converged > 0" requirement any more: a pair can make money on forced
                # exits alone, and score is signed now so the losers simply rank last
                if (d.get("total", 0) >= best_total_min
                        and (min_rate is None or (d.get("rate") or 0.0) >= min_rate)
                        and (min_win_rate is None or (d.get("win_rate") or 0.0) >= min_win_rate)
                        and d.get("score") is not None):
                    h = best_heaps[c]
                    item = (d["score"], a_n, b_n, bench, d.get("rate"), d.get("rate_lb"),
                            d.get("win_rate"), d.get("win_rate_lb"), d.get("cap_mean"),
                            d.get("alpha"), d.get("total"))
                    if len(h) < top_k_best:
                        heapq.heappush(h, item)
                    elif item[0] > h[0][0]:
                        heapq.heapreplace(h, item)
            rows.append(row)
            pairs_written += 1

        if rows:
            pd.DataFrame(rows, columns=summary_cols).to_csv(
                output_summary_csv, mode="a", header=False, index=False)
        print(f"  [{bench}] pairs kept={len(rows):,}  elapsed={time.time()-tb0:.1f}s")
        del pair_stats
        gc.collect()

    with _open_gz(output_best_pairs_jsonl, "wt") as bf:
        from datetime import datetime as _dtm
        bf.write(json.dumps({"meta": {
            "version": "pairflux_v1",
            "generated_at": _dtm.utcnow().isoformat() + "Z",
            "ranked_by": "score = conv_per_day — completed diverge->converge cycles per session",
        }}) + "\n")
        for c in CLASSES:
            top = sorted(best_heaps[c], key=lambda x: -x[0])
            bf.write(json.dumps({"cls": c, "top": [
                {"a": a, "b": b, "bench": bn, "score": _js(s), "rate": _js(r),
                 "rate_lb": _js(rl), "win_rate": _js(wr), "win_rate_lb": _js(wrl),
                 "cap_mean": _js(cm), "alpha": _js(sg), "total": t}
                for (s, a, b, bn, r, rl, wr, wrl, cm, sg, t) in top
            ]}, ensure_ascii=False) + "\n")

    onefile_f.close()
    if episodes_f is not None:
        episodes_f.close()

    # Only now do the finished files take their real names.
    _publish_staged()

    _f_onefile, _f_summary, _f_best, _f_episodes = _final_names
    if ROLL:
        print(f"rollover guard {list(ROLL)} min (+{ROLL_GUARD} min): runs trimmed={ROLL_STATS['runs_trimmed']:,} "
              f"dropped={ROLL_STATS['runs_dropped']:,}  episodes cut at a rollover={ROLL_STATS['episodes_cut']:,} "
              f"of {ROLL_STATS['episodes']:,}")
    print(f"DONE PairFlux pairs={pairs_written:,} elapsed={time.time()-t0:.1f}s")
    print(f"  onefile    = {_f_onefile}")
    print(f"  summary    = {_f_summary}")
    print(f"  best_pairs = {_f_best}")
    print(f"  episodes   = {_f_episodes if write_episodes else '(disabled)'}")

In [ ]:
from pathlib import Path
import os


def _resolve_orion_paths(strategy_code: str):
    final_env = os.environ.get("FINAL_PARQUET_PATH")
    sig_env   = os.environ.get("SIGNALS_DIR")
    orion_env = os.environ.get("ORION_HOME")

    final_path   = Path(final_env).expanduser().resolve() if final_env else None
    signals_base = Path(sig_env).expanduser().resolve()   if sig_env   else None

    if (final_path is None or signals_base is None) and orion_env:
        orion_home = Path(orion_env).expanduser().resolve()
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    if final_path is None or signals_base is None:
        here = Path.cwd().resolve()
        orion_home = None
        for parent in [here] + list(here.parents):
            if parent.name.lower() == "orion":
                orion_home = parent
                break
            cand = parent / "OriON"
            if cand.exists() and cand.is_dir():
                orion_home = cand.resolve()
                break

        if orion_home is None:
            raise RuntimeError("Cannot locate OriON. Set ORION_HOME env var (recommended).")

        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    out_dir = (signals_base / strategy_code.lower()).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    if not final_path.exists():
        raise FileNotFoundError(f"FINAL parquet not found: {final_path}")

    return final_path, out_dir


# ── runner ────────────────────────────────────────────────────────────────────

FINAL_PATH, OUT_DIR = _resolve_orion_paths("pairflux")

# The stage dir must live OUTSIDE signals/. run_orion_daily.py publishes the whole signals/ tree to
# the results repo, ignoring only *.tmp and *.lock — so a _stage folder under signals/pairflux/
# means ~300 MB of intermediate shuffle parquet gets committed and pushed on every daily run.
# OUT_DIR is <orion_home>/signals/pairflux, so this lands at <orion_home>/_work/pairflux_stage,
# which the publisher never looks at.
STAGE_DIR = OUT_DIR.parent.parent / "_work" / "pairflux_stage"
STAGE_DIR.parent.mkdir(parents=True, exist_ok=True)

# Stage 1 is the slow part and only depends on the class windows / date range, so it is skipped
# whenever a usable stage is already on disk — but it is NOT hard-coded off.
#
# Hard-coding RUN_STAGE1 = False broke the scheduled run twice: the stage lives outside signals/
# (so the publisher cannot ship 275 MB of intermediate parquet), and the machine that runs the
# daily job had never built it, so there was nothing to read. Deciding from the filesystem makes a
# fresh checkout self-healing while keeping the fast path on a machine that already has the stage.
# A stage that exists is only reusable while it is at least as new as final.parquet. Deciding on
# "does a stage exist" alone froze PairFlux at whatever day the stage was first built: measured
# 2026-09-20, the published PairFlux rolling_perf ended 2026-08-07 while Arbitrage, reading the SAME
# final.parquet directly, ended 2026-08-28 - three weeks of sessions never reached the stage, so
# every Scout window (5D included) showed old dates. The stage only depends on the class windows /
# date range, so re-staging when final.parquet is newer is always safe; it just costs the slow pass.
def _stage_is_current(stage_dir, final_path):
    files = list(stage_dir.glob("*.parquet")) if stage_dir.exists() else []
    if not files:
        return False
    return min(f.stat().st_mtime for f in files) >= final_path.stat().st_mtime

RUN_STAGE1 = not _stage_is_current(STAGE_DIR, FINAL_PATH)
print(f"stage dir: {STAGE_DIR}  ->  {'BUILDING (stage 1)' if RUN_STAGE1 else 'reusing existing'}")

if RUN_STAGE1:
    pairflux_stage1_shuffle(
        input_path=str(FINAL_PATH),
        stage_dir=str(STAGE_DIR),
        class_windows=CLASS_WINDOWS_DEFAULT,
        session_split_min=1020,     # 17:00 — everything later belongs to the next session
        start_date=None,            # e.g. "2026-01-01" to cut history and memory
        bench_whitelist=None,       # e.g. ["SPY", "IWM"] to test on two groups first
        STOCK_NUM_FIELD="Stack%",
    )

pairflux_stats_exporter(
    stage_dir=str(STAGE_DIR),
    output_onefile_jsonl=str(OUT_DIR / "onefile.jsonl.gz"),
    output_summary_csv=str(OUT_DIR / "summary.csv"),
    output_best_pairs_jsonl=str(OUT_DIR / "best_pairs.jsonl.gz"),
    output_episodes_jsonl=str(OUT_DIR / "episodes.jsonl.gz"),
    write_episodes=True,
    class_windows=CLASS_WINDOWS_DEFAULT,
    # OPEN rates only the deviations BORN in 9:00-9:25, but still gives them until 10:00
    # to normalise. PRE/INTRA count onsets anywhere inside their own window.
    onset_windows={"OPEN": ((9, 0), (9, 25))},
    # EVERY class counts a divergence that was ALREADY open when its window began. Such an
    # episode has an unknown birth time — it may have gapped in overnight — but it is a real,
    # tradeable spread the moment the window opens, and that is what the statistics are for.
    # Excluding them threw away 52% of INTRA episodes and 73% of everything peaking above 2pp,
    # while the convergence rate of the excluded ones was no worse (64.7% vs 63.8%).
    # OPEN still requires the episode to START inside 09:00-09:25 — that is its onset window,
    # a separate rule from this one, and it stays.
    require_fresh_onset=False,
    session_split_min=1020,
    bar_minutes=None,               # infer from data
    hedge_mode="ols",               # "unit" = plain Stack%_A - Stack%_B
    # Divergence strength is measured in PERCENTAGE POINTS, and the thresholds are PER CLASS.
    # div_z/conv_z=None turns the sigma test off entirely — if a class floods you with episodes
    # from pairs whose ordinary noise is already as wide as its threshold, put div_z=1.5 back to
    # require the move be unusual for THAT pair too.
    #
    # div_abs_pp DEFINES WHAT COUNTS AS AN EVENT — it is not a filter on how big the episode
    # later grows. A pair has diverged once the spread reaches the class threshold, and
    # everything downstream (total, rate, alpha, sigma, conv_per_day) is the statistics OF THOSE EVENTS.
    # The separate peak-size filters are min_abs_peak_pp / max_abs_peak_pp, and both stay 0 = off,
    # so no episode is dropped for how far it ran.
    #
    # The deviation is measured with each pair's OWN beta for THIS class, so a point of threshold
    # means a point of genuine dislocation, not a point of the two legs simply being geared
    # differently. Audited on the staged 3 months: the published beta reproduces to 2e-4 and
    # agrees with the median per-day slope on 99% of pairs.
    #
    # WHY INTRA RUNS AT 1.0 / 0.3 AND THE OTHERS DO NOT (measured 2026-09-06 over the same 64
    # staged sessions; the 0.5 row reproduces the published run episode for episode):
    #        INTRA, 10:00, 0.5/0.2   19 162 pairs   445 423 episodes   rate 64.7%   alpha 0.72pp
    #                                median 21 episodes/pair   EV +0.0591 pp/episode
    #        INTRA, 09:45, 1.0/0.3   15 211 pairs   269 474 episodes   rate 64.5%   alpha 1.32pp
    #                                median 15 episodes/pair   EV +0.0940 pp/episode
    # Expected capture per episode rises 59% because the payoff scales with the entry level while
    # the exit stays fixed. The convergence RATE barely moves (64.7 -> 64.5): a wider entry does
    # not make a pair likelier to come back, it makes the trip worth more when it does. Ranking on
    # rate would therefore see no difference at all here — rank on cap_lb.
    # PRE and OPEN keep 0.5 / 0.2: at 13.1k and 2.4k episodes they are far smaller classes, their
    # EV is already 4-6x INTRA's, and raising their bar leaves too little to rate.
    #
    # The floor on a converged episode's capture is (div - conv): 0.7pp for INTRA, 0.3pp elsewhere.
    div_z=None,
    div_abs_pp={"PRE": 0.5, "OPEN": 0.5, "INTRA": 1.0},
    conv_z=None,
    conv_abs_pp={"PRE": 0.2, "OPEN": 0.2, "INTRA": 0.3},
    min_hold=3,
    rollover_minutes=(0, 240), rollover_guard_min=10,   # feed rollover 00:00/04:00: cut episodes there, guard births (see the exporter)
    scale_mode="std",               # switch to "mad" if a class comes back empty
    # zero = the pair's TYPICAL state (median-centred), so a divergence is measured from
    # where the pair normally sits. "zero" instead measures from literal parity A - beta*B.
    center_mode="auto",
    # measured on real data: overnight/pre-market bars are 2-4 min apart, so a strict
    # 1-minute adjacency rule prevents PRE/OPEN episodes from ever forming
    max_gap_minutes=5,
    min_abs_peak_pp=0.0,            # e.g. 0.3 to ignore untradeably small divergences
    max_abs_peak_pp=0.0,            # e.g. 15.0 to drop news-driven pseudo-divergences
    min_corr=0.7, corr_step_bars=5,
    max_pairs_per_bench=20000,
    min_bars_per_ticker=500, min_days_per_ticker=10,
    max_tickers_per_bench=800, max_matrix_mb=2000,
    # None = start the sigma walk at each class's own div_abs_pp, which is the only floor that
    # means anything: below the entry threshold there are no episodes to measure.
    sigma_min_pp=None,
    # GAMMA floor. INTRA 4.0 is the measured setting; PRE and OPEN start at 3.0 because their
    # episodes are far fewer and a higher floor leaves nothing to qualify. None disables gamma.
    # GAMMA is now measured for all three classes (2026-09-13, 3.96M simulated trades, three
    # expanding-window blocks of 10-11 sessions each, settings chosen on the WORST block and only
    # among those with at least 300 trades - the unconstrained winners sat on 20-25 trades):
    #
    #   PRE    floor 11, exit 0.50, entries until 09:00, z 1.96, n>=5
    #          611 trades (21/day), 85.8% profitable, +9.25 pp/trade, blocks +9.8 +11.6 +7.2
    #   OPEN   floor 2.5, exit 0.75, entries until 09:25, z 1.00, n>=10
    #          576 trades (20/day), 69.6% profitable, +0.69 pp/trade, blocks +0.46 +1.14 +0.46
    #   INTRA  floor 12, exit 0.75, entries until 13:00, z 1.96, n>=10
    #          unchanged - an independent split reproduced it (100 trades, 79.0%, +2.40)
    #
    # PRE'S NUMBERS DESERVE SUSPICION AND ARE NOT YET EARNED. Its entries are confined to
    # 21:03-02:00, the thinnest part of the pre-market, where a leg's Stack% can sit on a stale
    # last print for hours. A "divergence" that reverts by 09:30 may just be that leg catching up
    # once real trading resumes, at a price nobody could have entered. Untested either way.
    gamma_floor_pp={"PRE": 11.0, "OPEN": 2.5, "INTRA": 12.0},
    gamma_step_pp={"PRE": 1.0, "OPEN": 0.5, "INTRA": 1.0},
    gamma_exit_ratio={"PRE": 0.50, "OPEN": 0.75, "INTRA": 0.75},
    # PRE takes entries across its whole evening-to-morning stretch, 21:00-09:00, by your
    # rule rather than by the sweep - the sweep preferred 02:00 only because it was hunting
    # a maximum. Re-measured under 09:00 with the artefact cap in place, floor 11 gives 611
    # trades (21/day), 85.8% profitable, +9.25 pp/trade, worst block +7.17.
    gamma_cutoff_hm={"PRE": (9, 0), "OPEN": (9, 25), "INTRA": (13, 0)},
    # 1.96 demands the 95% lower bound on capture, not just a positive mean. That is the setting
    # validated across three different train/test splits: ~5-8 signals a day, 58-67% of them
    # profitable, +1.39 to +1.91 pp per trade, the confidence interval positive in every split.
    # 0.0 would widen it to ~720 pairs but drop the edge to +0.71 pp/trade.
    gamma_z={"PRE": 1.96, "OPEN": 1.00, "INTRA": 1.96},
    gamma_min_total={"PRE": 5, "OPEN": 10, "INTRA": 10},
    min_total=3,                    # 3 episodes is a low evidence bar — lean on rate_lb, not rate
    min_rate=0.3,                   # keep a class if it converges at least 30% of the time
    best_min_total=3,               # same bar as the CSV now
    beta_band=None,                 # 1.5 keeps only genuinely 1:1 pairs (drops geared ETFs)
    top_k_best=500,
    compute_adf=False,              # see the note in the docstring before turning this on
)


In [ ]:
from __future__ import annotations

# ------------------------------------------------------------------
# PAIRFLUX ROLLING RECENCY PERFORMANCE (last 5/20/40/65 sessions) + P&L TRADE LOG, split by
# class x direction x entry-deviation bucket. Feeds the "Scout" UI exactly like ArbitRage's
# devsig_rolling_perf_exporter (notebooks/ArbitRage.ipynb) - same bucket grid, same rolling-window
# scheme, same trade-log shape - adapted for a PAIR instead of a single ticker.
#
# UNLIKE ArbitRage's exporter, this one does NOT re-scan final.parquet: pairflux_stats_exporter
# (above) already runs the full episode-detection machine once per (pair, class) and publishes
# every episode to episodes.jsonl(.gz) - entry_dev, exit_dev, capture, entry_minute, exit_minute,
# kind, dir, all already computed. This function is a second, MUCH cheaper pass that just re-buckets
# that published trade log by recency window and entry-deviation bucket, the same way ArbitRage's
# rolling_perf re-buckets its own tick-level scan. Run it any time after episodes.jsonl - it never
# reopens final.parquet and does not need stage_dir.
def pairflux_rolling_perf_exporter(
    episodes_path: str,
    output_path: str = "PAIRFLUX/rolling_perf.json.gz",
    *,
    windows_days=(5, 20, 40, 65),
    # bucket the ENTRY deviation (entry_dev, at the CONFIRMED entry row) - no lookahead, this is
    # what you would actually know the instant you opened the trade. Peak is known only after the
    # fact and would leak information a live entry never had, exactly as in ArbitRage.
    start_bin_step: float = 0.5,
    start_bin_max: float = 8.0,
    min_events_per_pair: int = 10,
    min_events_per_cell: int = 2,
    # $ position size assumed for the SPREAD (both legs together, equal-notional) when turning an
    # episode's capture (percentage points) into P&L - see PairFlux's own "equal-notional legs"
    # convention (project_pairflux memory): $1000 total behaves like $1000 moving `capture` pp.
    position_usd: float = 1000.0,
    include_trade_log: bool = True,
    log_every_n_lines: int = 500_000,
):
    """
    Rolling recency performance for PairFlux, split by class x direction x entry-deviation bucket
    x trailing window (default last 5/20/40/65 sessions), PLUS a raw per-episode trade log for
    P&L-curve rendering - the PairFlux analogue of ArbitRage's rolling_perf.json.gz, read by the
    same "Scout" UI pattern: pick a pair/class/direction/window, set an entry-deviation threshold,
    get a hit-rate and a $ P&L curve, entirely client-side against this one file.

    WHY ENTRY, NOT PEAK: bucketing on entry_dev (the deviation at the CONFIRMED entry - min_hold
    candles past the divergence threshold, see pairflux_stats_exporter's _episodes) means every
    bucket only ever contains what you could have known the moment you opened the trade. Bucketing
    on `peak` instead would let a trade that entered at 0.6pp and later widened to 2.1pp count as a
    "2.0-2.5" signal, which is exactly the lookahead ArbitRage's own rolling_perf avoids.

    WHY WINDOWS ARE FILE-RELATIVE: episodes.jsonl is a static historical export, not a live feed,
    so "last 5/20/40/65 days" means the 5/20/40/65 most recent SESSION DATES actually present in
    the file (per each episode's `date` field), not datetime.date.today(). A session's rank is 0
    for the most recent date, 1 for the one before it, etc.; an episode's date qualifies for
    window W iff its rank < W. Windows are cumulative (last-5 subset-of last-20 subset-of last-40
    subset-of last-65). The trade log carries every episode within the LARGEST window (default
    65); the frontend slices it down to 5/20/40 using meta.recent_dates, exactly as ArbitRage's
    Scout already does - the backend never emits four overlapping copies of the same trades.

    P&L MODEL (real spread capture, from the episode log - not re-derived here):
      - `capture` (percentage points, signed positive = profit) is taken AS-IS from
        episodes.jsonl: entry (confirmed) -> exit (confirmed convergence, or the class window's
        last candle of that session if it never converged). PairFlux's exit time is NOT a fixed
        clock time the way ArbitRage's is - it is genuinely per-episode data (early on a fast
        convergence, late on a forced exit), which is why exit_minute is read from the file
        rather than computed from a class-close constant.
      - direction: `dir` > 0 means the A leg ran ahead of the model's B-implied value at entry -
        the trade that converges it is SHORT A / LONG B ("pos"); `dir` < 0 is LONG A / SHORT B
        ("neg"). Equal-notional legs are assumed (see project_pairflux memory): pnl_usd treats
        `position_usd` as the total notional deployed across BOTH legs, so
        pnl_usd = position_usd * capture / 100 exactly mirrors ArbitRage's own pnl_usd formula.

    OUTPUT: a single gzip-if-.gz-suffixed JSON file:
      {"meta": {..., "recent_dates": [...], "position_usd": 1000.0, "classes": [...]},
       "rows": [ {a, b, bench,
                   classes: {INTRA: {pos: {"0.50-1.00": {"w5": {...}, ...}}}}},
                   trades: [{cls, sign, date, entry_dev, peak, capture, kind,
                             entry_minute, exit_minute}, ...] (chronological) },
                  ... ]}
    Sparse: a (class, direction, bucket, window) cell is only present if it has at least one
    episode; a cell's rate/avg_pnl fields are null (raw counts kept) below min_events_per_cell.
    """
    import gzip, json, time
    from collections import Counter, defaultdict
    from pathlib import Path

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    WINDOWS = tuple(sorted(windows_days))
    MAX_WINDOW = WINDOWS[-1]

    def _open_text(path: str, mode: str = "rt"):
        p = str(path).lower()
        if p.endswith(".gz"):
            return gzip.open(path, mode, encoding="utf-8")
        return open(path, mode, encoding="utf-8")

    def _open_out(path: str, mode: str = "wt"):
        if str(path).lower().endswith(".gz"):
            return gzip.open(path, mode, encoding="utf-8", newline="\n", compresslevel=6)
        return open(path, mode.replace("t", ""), encoding="utf-8", newline="\n")

    def start_bin_label(abs_dev: float) -> str:
        if abs_dev >= start_bin_max:
            return f"{start_bin_max:.1f}+"
        idx = int(abs_dev / start_bin_step + 1e-9)
        lo = idx * start_bin_step
        hi = lo + start_bin_step
        return f"{lo:.2f}-{hi:.2f}"

    # ---------------- pass 1: session-date ranking (file-relative windows) ----------------
    t0 = time.time()
    seen_dates = set()
    classes_seen = set()
    n_lines_p1 = 0
    with _open_text(episodes_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            n_lines_p1 += 1
            ep = json.loads(line)
            seen_dates.add(ep["date"])
            classes_seen.add(ep["cls"])
            if log_every_n_lines and n_lines_p1 % log_every_n_lines == 0:
                print(f"[pass 1] lines={n_lines_p1:,} dates={len(seen_dates)} elapsed={time.time()-t0:.1f}s")

    all_dates_sorted = sorted(seen_dates)  # ISO date strings sort correctly, ascending
    rank_from_end = {d: i for i, d in enumerate(reversed(all_dates_sorted))}  # 0 = most recent
    n_sessions = len(all_dates_sorted)
    recent_dates = all_dates_sorted[-MAX_WINDOW:]
    CLASSES = sorted(classes_seen)

    def windows_for_date(date_str: str):
        r = rank_from_end.get(date_str)
        if r is None:
            return ()
        return tuple(w for w in WINDOWS if r < w)

    print(f"[pairflux rolling_perf] sessions found: {n_sessions} "
          f"(most recent: {all_dates_sorted[-1] if all_dates_sorted else None}), "
          f"classes={CLASSES}, windows={WINDOWS}")

    # ---------------- pass 2: per-pair aggregation ----------------
    def new_agg_pair():
        return {c: {sk: defaultdict(lambda: defaultdict(
                        lambda: {"c": Counter(), "dev_sum": 0.0, "dev_n": 0, "cap_sum": 0.0, "cap_n": 0}))
                     for sk in ("pos", "neg")} for c in CLASSES}

    agg = {}
    bench_of = {}
    trades = {}  # (a,b) -> list of trade dicts (within the largest window only)
    pair_total_in_max = Counter()

    t1 = time.time()
    n_lines_p2 = 0
    with _open_text(episodes_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            n_lines_p2 += 1
            ep = json.loads(line)
            wins = windows_for_date(ep["date"])
            if not wins:
                continue

            key = (ep["a"], ep["b"])
            cls = ep["cls"]
            sk = "pos" if ep["dir"] > 0 else "neg"
            abs_entry = abs(float(ep["entry_dev"]))
            bucket = start_bin_label(abs_entry)
            status = ep.get("kind") or "none"
            capture = ep.get("capture")

            bench_of.setdefault(key, ep.get("bench"))
            a = agg.setdefault(key, new_agg_pair())
            for w in wins:
                cell = a[cls][sk][bucket][f"w{w}"]
                cell["c"][status] += 1
                cell["dev_sum"] += abs_entry
                cell["dev_n"] += 1
                if capture is not None:
                    cell["cap_sum"] += float(capture)
                    cell["cap_n"] += 1
            pair_total_in_max[key] += 1

            if include_trade_log:
                pnl_usd = None if capture is None else round(position_usd * float(capture) / 100.0, 2)
                trades.setdefault(key, []).append({
                    "cls": cls, "sign": sk, "date": ep["date"],
                    "entry_dev": round(abs_entry, 4),
                    "peak": None if ep.get("peak") is None else round(float(ep["peak"]), 4),
                    "capture": None if capture is None else round(float(capture), 4),
                    "pnl_usd": pnl_usd,
                    "status": status,
                    "entry_minute": ep.get("entry_minute"),
                    "exit_minute": ep.get("exit_minute"),
                })

            if log_every_n_lines and n_lines_p2 % log_every_n_lines == 0:
                print(f"[pass 2] lines={n_lines_p2:,} pairs={len(agg):,} elapsed={time.time()-t1:.1f}s")

    # ---------------- build sparse output ----------------
    def cell_summary(cell):
        c = cell["c"]
        total = int(sum(c.values()))
        hard = int(c.get("hard", 0)); soft = int(c.get("soft", 0)); none = int(c.get("none", 0))
        any_ = hard + soft
        enough = total >= min_events_per_cell
        return {
            "total": total, "hard": hard, "soft": soft, "none": none,
            "rate_any": (any_ / total) if (total and enough) else None,
            "rate_hard": (hard / total) if (total and enough) else None,
            "rate_soft": (soft / total) if (total and enough) else None,
            "avg_entry_dev": round(cell["dev_sum"] / cell["dev_n"], 4) if cell["dev_n"] else None,
            "avg_capture_pp": round(cell["cap_sum"] / cell["cap_n"], 4) if (cell["cap_n"] and enough) else None,
            "avg_pnl_usd": round(position_usd * (cell["cap_sum"] / cell["cap_n"]) / 100.0, 2) if (cell["cap_n"] and enough) else None,
        }

    rows = []
    for key, cls_map in agg.items():
        classes_out = {}
        for cls in CLASSES:
            sign_out = {}
            for sk in ("pos", "neg"):
                buckets = cls_map[cls][sk]
                if not buckets:
                    continue
                bucket_out = {}
                for bucket, win_map in buckets.items():
                    win_out = {wkey: cell_summary(cell) for wkey, cell in win_map.items()}
                    bucket_out[bucket] = win_out
                if bucket_out:
                    sign_out[sk] = bucket_out
            if sign_out:
                classes_out[cls] = sign_out

        if pair_total_in_max[key] < int(min_events_per_pair):
            continue

        a_n, b_n = key
        row = {"a": a_n, "b": b_n, "bench": bench_of.get(key), "classes": classes_out}
        if include_trade_log:
            row["trades"] = sorted(trades.get(key, []), key=lambda r: (r["date"], r["cls"]))
        rows.append(row)

    payload = {
        "meta": {
            "version": "pairflux-rolling-perf-v1",
            "generated_at": __import__("datetime").datetime.utcnow().isoformat() + "Z",
            "windows_days": list(WINDOWS),
            "sessions_seen": n_sessions,
            "most_recent_session": all_dates_sorted[-1] if all_dates_sorted else None,
            "recent_dates": recent_dates,
            "classes": CLASSES,
            "start_bin_step": start_bin_step, "start_bin_max": start_bin_max,
            "bucketed_on": "entry_dev (confirmed-entry deviation, pp, no lookahead)",
            "min_events_per_pair": min_events_per_pair,
            "min_events_per_cell": min_events_per_cell,
            "position_usd": position_usd,
            "pnl_basis": "capture (pp) from episodes.jsonl, entry(confirmed)->exit(convergence or forced end-of-day); equal-notional legs",
        },
        "rows": rows,
    }

    with _open_out(output_path, "wt") as f:
        json.dump(payload, f, ensure_ascii=False)

    print(f"DONE pairflux_rolling_perf lines_read={n_lines_p2:,} pairs_out={len(rows):,} -> {output_path}")


In [ ]:
# ------------------------------------------------------------------
# ROLLING RECENCY PERFORMANCE (last 5/20/40/65 sessions) + P&L TRADE LOG for the "Scout" UI -
# the PairFlux analogue of ArbitRage's rolling_perf.json.gz (notebooks/ArbitRage.ipynb, same
# cell pair). Reads episodes.jsonl.gz just published above; does NOT touch final.parquet or
# stage_dir, so it can be re-run on its own any time the bucket grid needs retuning.
# ------------------------------------------------------------------

ROLLING_OUT_PATH = OUT_DIR / "rolling_perf.json.gz"

pairflux_rolling_perf_exporter(
    episodes_path=str(OUT_DIR / "episodes.jsonl.gz"),
    output_path=str(ROLLING_OUT_PATH),

    windows_days=(5, 20, 40, 65),

    # entry_dev scale differs from ArbitRage's sigma-based buckets: PRE/OPEN divergences start
    # at 0.5pp, INTRA at 1.0pp, and the measured GAMMA floors reach 11-12pp - widen start_bin_max
    # if the far tail (rare wide divergences) needs its own buckets.
    start_bin_step=0.5,
    start_bin_max=8.0,

    min_events_per_pair=10,
    min_events_per_cell=2,

    position_usd=1000.0,
    include_trade_log=True,
)

print("PairFlux rolling performance + P&L export completed ->", ROLLING_OUT_PATH)
